# Vehicle Attrition as a Signal for Emerging Vehicle Collectability

**Author:** Phillip Smith\
**Course:** Machine Learning & AI\
**Term:** Fall 2026\
**Employer Partner:** Hagerty

---

## Notebook Overview

This notebook covers the exploratory data analysis (EDA), feature engineering, and model development and evaluation phases of a semester-long capstone project investigating vehicle attrition patterns and their potential as early signals of emerging vehicle collectability.

The analysis draws on two integrated datasets — VIO (Vehicles in Operation) registration snapshots and VINtelligence vehicle specification data — unified into a single cohort-level table in Snowflake (`WORKSPACE.ENGINEER.VIO_COHORT_ENRICHED`). All data preparation and integration work was completed prior to this notebook; it is summarized in the Data Preparation section below and documented in the project decision log.

**Scope:** cars and trucks from model year 1981 onward with VINtelligence specifications (14,339 cohorts), including light and medium duty commercial vehicles. Specifications are central to identifying enthusiast vehicles and to feature engineering, and VINtelligence begins with model year 1981.

**Pre-1981 cohorts and motorcycles are prepared in Snowflake but excluded from this analysis.**

---

## Business Context

Hagerty is a specialty automotive lifestyle and insurance company focused on classic and collector vehicles. As a production vehicle batch ages, its registered population declines. Year-over-year shifts in that attrition rate may serve as an early signal of emerging collectability — as supply shrinks, scarcity can pressure values upward and slow further attrition, creating a feedback loop between scarcity and desirability. Early identification of these signals could enable Hagerty to identify emerging enthusiast markets before they mature.

---

## Research Questions

**RQ1 —** What are the representative attrition patterns of vehicles characterized as regular transportation versus enthusiast-oriented, based on vehicle specification attributes?\
**RQ2 —** Can annual vehicle attrition be predicted for a given vehicle descriptor, and can the model accurately predict attrition across multiple annual periods?

---

## Hypothesis

Vehicles that exhibit a deceleration in annual attrition rate are candidates for emerging collectability. As collectors begin preserving a vehicle, scrapping slows, registered population stabilizes, and the attrition rate declines - a behavioral shift that may appear in registration data before the broader market recognizes the vehicle's collector status.

This analysis investigates whether that signal is detectable and consistent, and whether it is reinforced by structural vehicle characteristics such as drivetrain, body style, engine displacement, and transmission type. Vehicles showing both behavioral deceleration and enthusiast-aligned specifications represent the strongest candidates for emerging collectability.

The hypothesis is investigative, not assumed. The data will determine whether the signal exists, how early it appears, and whether it differs meaningfully across vehicle types.

---

## Data Preparation (Snowflake)

All preparation was completed in Snowflake before this notebook, which reads only the final table. The major steps:

1. **Loading the raw data.** Uploaded 25 compressed VIO files and the VINtelligence file to a Snowflake stage with Python, then loaded them into raw tables. The VIO files came in five different layouts, so each layout was loaded with `COPY INTO` into its own table (`VIO_RAW_TYPE1` to `VIO_RAW_TYPE5`), adding each file's snapshot date from its file name. VINtelligence was loaded as one row per VIN pattern (379,960 patterns, 636 columns) into `VINTELLIGENCE_RAW`.
2. **Standardizing VIO.** Built the `VIO_UNIFIED` view to map the five layouts to one schema (for example, `COUNT`, `TOTAL_COUNT`, and `TOTAL_CNT` all become `VIO_COUNT`). It holds 1.93 billion rows: 20 quarterly car and truck snapshots (April 2021 to July 2026) and 5 annual motorcycle and powersports snapshots (January 2022 to January 2026).
3. **Matching makes.** VIO records full make names, while VINtelligence uses make codes. `MAKE_CROSSWALK` (159 mappings) links them by exact match on VINtelligence's full make name. It replaced a first version built on prefix matching, which assigned wrong codes to several makes (e.g. Fisker, Royal Enfield, Triumph Motorcycle) and missed others (e.g. Bugatti Rimac).
4. **Filtering to matchable records.** `VIO_MATCHED` keeps VIO records whose make is in the crosswalk and whose model name appears in VINtelligence. Only exact matches are used, so unmatched records are excluded rather than guessed; the 426 unmatched model names are saved in `VIO_EXCLUDED_MODELS` for review. For model years 1981 onward, this excluded 0.4% of car and truck records (0.2% of registered vehicles).
5. **Summarizing specifications by model.** VIO has no VINs, so the finest level both sources share is model year, make, and model (a cohort). `VIN_MODEL_AGG` (27,997 models) summarizes each model's VIN patterns: lowest base MSRP; largest displacement, cylinder count, weight, and door count; most common drive type, body style, fuel type, segment, and origin; whether a manual transmission or rear-wheel drive was offered; and the number of distinct configurations.
6. **Correcting data quality issues.** Checking these summaries against the raw VINtelligence file found and fixed several problems:
   - Numeric specifications were stored as text, so minimum and maximum values were compared alphabetically and were wrong for about 6% of MSRPs (and smaller shares of displacement, cylinder, and weight values). Values are now converted to numbers before summarizing.
   - The rear-wheel drive flag looked for the code `R`, but the data uses `RWD`, so the flag was always 0.
   - The manual flag ignored optional transmissions, and both flags recorded 0 when the source was blank. Blanks are now recorded as missing (unknown) rather than "no."
   - Zero placeholders in MSRP and weight are treated as missing.
   - The VIN pattern count reflected how each manufacturer encodes VINs rather than how many versions a model had (e.g. 2015 Volvo V60: 1,032 VIN patterns but 26 configurations), so it was replaced with a configuration count.
7. **Building the analysis table.** `VIO_COHORT` sums `VIO_MATCHED` to one row per cohort per snapshot across all states, counties, and zip codes. `VIO_COHORT_ENRICHED` adds the specifications with a left join, so cohorts without specifications are kept rather than dropped. The final table has the same row count as `VIO_COHORT` (362,343), and total registrations (5.78 billion, summed across snapshots) are identical in `VIO_MATCHED`, `VIO_COHORT`, and `VIO_COHORT_ENRICHED`, confirming that aggregation and the join neither lost nor duplicated records.
8. **Scope for this notebook.** The notebook keeps cars and trucks from model year 1981 onward with specifications (14,339 cohorts). Motorcycles and pre-1981 cohorts remain available in Snowflake.

---

## Analysis Dataset: Tables, Key Columns & Limitations

| Table | Rows | Description |
| --- | --- | --- |
| `VIO_COHORT_ENRICHED` | 362K | Primary analysis table — cohort-level registration counts by snapshot with VINtelligence specs joined |
| `VIN_MODEL_AGG` | 28K | VINtelligence specs aggregated to model grain (model year + make + model) |
| `VIO_MATCHED` | 1.9B | Cleaned and filtered VIO registration records |
| `MAKE_CROSSWALK` | 159 | VIO make name to VINtelligence make code, matched on VINtelligence's full make name |

**Key columns in `VIO_COHORT_ENRICHED`:**

| Column | Type | Description |
| --- | --- | --- |
| `YEAR_MODEL` | Integer | Vehicle model year |
| `MAKE` | String | Vehicle make |
| `MODEL` | String | Vehicle model |
| `VINT_MAKE_CD` | String | VINtelligence make code (join key) |
| `SNAPSHOT_DATE` | Date | Observation date — quarterly snapshots from April 2021 to July 2026 (July and October 2021 are not available) |
| `VIO_COUNT` | Integer | Registration count for this cohort at this snapshot |
| `BASE_MSRP` | Float | Lowest base MSRP across the model's VIN patterns (zero placeholders treated as missing) |
| `MAX_ENGINE_DISP` | Float | Largest engine displacement available (cubic inches) |
| `MAX_CYLINDERS` | Integer | Largest cylinder count available |
| `HAS_MANUAL` | Flag (0/1) | Manual transmission offered, as the primary or an optional transmission |
| `HAS_RWD` | Flag (0/1) | Rear-wheel drive offered (excludes rear-wheel drive with 4x4) |
| `PRIMARY_DRIVE_TYPE` | String | Most common drive type across VIN patterns |
| `PRIMARY_BODY_STYLE` | String | Most common body style across VIN patterns |
| `MAX_DOOR_CNT` | Integer | Maximum door count available |
| `PRIMARY_VEH_TYPE` | String | Most common vehicle type classification |
| `PRIMARY_FUEL_TYPE` | String | Most common fuel type |
| `SEGMENTATION` | String | Most common market segment classification |
| `MAX_SHIP_WEIGHT` | Float | Largest shipping weight in pounds (zero placeholders treated as missing) |
| `ORIGIN` | String | Most common vehicle origin |
| `CONFIG_COUNT` | Integer | Number of distinct configurations (trim, body style, displacement, drive type, transmission) offered for the model; a measure of variety |
| `VIO_VEHICLE_TYPE` | String | Vehicle type recorded in VIO (car or light truck); used to limit the analysis to cars and trucks |

"Most common" values are the mode across VIN patterns, not weighted by vehicles produced or registered. Ties between equally common values are broken by a fixed, neutral rule, so values are repeatable.

**Known limitations:**

- Pre-1981 cohorts (1,844) are excluded because VINtelligence has no specifications before model year 1981, along with 5 later cohorts with no specification match. As a result, established pre-1981 collector vehicles are not available as a reference for collectible attrition patterns.
- For model years 1981 onward, 0.4% of car and truck VIO records (0.2% of registered vehicles) were excluded during integration because the make or model had no exact match in VINtelligence.
- Pre-1981 data is also incomplete upstream: in the July 2026 snapshot, 27% of pre-1981 vehicles were excluded during integration, mainly records with no model name and classic models discontinued before VINtelligence begins (e.g. Volkswagen Type 1, MGB, Plymouth Barracuda), along with three makes (Austin Rover, Capri, Opel) that have no VINtelligence equivalent.
- VINtelligence provides no values for ISO performance indicator, advanced vehicle type, or EV range; these columns are dropped.
- Rotary engines and unknown cylinder counts are recorded as missing in `MAX_CYLINDERS`.
- Attrition is compared year over year using the same quarter in consecutive years. Because snapshots begin in April 2021 and July and October 2021 are missing, only April has a 2021 to 2022 comparison; the other quarters begin with 2022 to 2023.
- A small number of cohorts represent non-US-market vehicles with very low counts, such as Canada-only models (e.g. the 1985 to 1987 Hyundai Pony and Stellar, and the 2005 to 2006 Smart Fortwo sold before its US launch).
- The most recent model years are still being sold into the market, so their registration counts are growing rather than declining.

---

## Notebook Structure

1. **Environment Setup** — connect to Snowflake, import libraries
2. **Data Loading & Schema Review** — load `VIO_COHORT_ENRICHED`, confirm shape and types
3. **Exploratory Data Analysis** — cohort size and model year distributions, specification distributions by vehicle class, correlation matrix
4. **Update Since Milestone 2** — data preparation changes made after the exploratory analysis
5. **Data Quality Checks** — snapshot coverage, non-standard value formatting, outliers, missing values, sampling bias
6. **Feature Engineering** — attrition metrics (peak VIO, vehicle age, annual attrition rate, cumulative attrition, attrition acceleration) and specification features
7. **Era Split** — divide cohorts by model year era
8. **One-Hot Encoding** — convert the selected categorical columns to 0/1 columns, by era
9. **Variable Significance Analysis** — identify the features that matter most for annual and excess attrition, by era
10. **Summary of Features and Transformations** — features created, null value management, outlier treatment, sampling bias, non-standard value formatting
11. **Model Development and Evaluation** — attrition regression (RQ2), collectability classification, generalization to unseen model families, and vehicle clustering (RQ1)
12. **Model Selection Summary** — the selected model for each task, why it was selected, and limitations

## Environment Setup

In [ ]:
# Data manipulation
import re
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

# Snowflake
from snowflake.snowpark import Session

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.3f}'.format)
%matplotlib inline

# connection settings live in ~/.snowflake/connections.toml (outside the repo); sign in opens in the browser
session = Session.builder.config("connection_name", "vehicle_attrition").create()

## Data Loading & Schema Review

In [ ]:
df_cohort = session.table("WORKSPACE.ENGINEER.VIO_COHORT_ENRICHED").to_pandas()

df_cohort['SNAPSHOT_DATE'] = pd.to_datetime(df_cohort['SNAPSHOT_DATE'])
df_cohort['YEAR_MODEL'] = df_cohort['YEAR_MODEL'].astype(int)

# scope: cars and trucks only; motorcycle data retained upstream for future analysis
df_cohort = df_cohort[df_cohort['VIO_VEHICLE_TYPE'].isin(['CAR', 'LIGHT TRUCK'])]

# scope: model years 1981 onward with VINtelligence specs, since specs drive feature engineering;
# CONFIG_COUNT is present for every cohort matched to VINtelligence; pre-1981 cohorts are retained upstream
df_cohort = df_cohort[(df_cohort['YEAR_MODEL'] >= 1981) & df_cohort['CONFIG_COUNT'].notna()]

df_cohort.info()

## Exploratory Data Analysis

**Distribution**

- VIO Count Distribution: Tells you the range of population sizes across cohorts, ensuring attrition rates are compared fairly regardless of how common or rare a vehicle is.
- Model Year Distribution: Tells you how many cohorts exist per decade and where your data coverage is densest.
- Spec Column Distribution: Tells you whether your enthusiast-indicator variables have enough information to be useful. If majority of vehicles have the same body style code, that variable will not help the clustering algorithm distinguish anything.

**Correlation Matrix**

- Uses `cohort_specs` with peak VIO from `cohort_summary` merged in to build a combined dataframe called `corr_data`
- `corr_data` is built only for the correlation matrix: it has one row per cohort (no snapshot time series), with each cohort's peak VIO added

**Summary Statistics (Cars and Trucks, Model Years 1981 Onward, 14,339 cohorts) -**

- Model years range from 1981 to 2027 (median 2007).
- Every cohort has VINtelligence specifications, but availability varies by field (e.g. MSRP 13,484 and transmission 11,816 of 14,339 cohorts).
- Among cohorts with known data, 41% offered a manual transmission and \~50% offered rear-wheel drive.
- Specification ranges are plausible: cylinders reach 16 (Bugatti W16), displacement reaches 636 ci, and MSRP reaches $5.4M (Bugatti Mistral).
- Configuration count is right-skewed (median 3 for cars and 5 for trucks; maximum 153).

In [ ]:
spec_cols = ['BASE_MSRP', 'MAX_ENGINE_DISP', 'MAX_CYLINDERS', 'HAS_MANUAL', 'HAS_RWD',
             'PRIMARY_DRIVE_TYPE', 'PRIMARY_BODY_STYLE', 'MAX_DOOR_CNT', 'PRIMARY_VEH_TYPE',
             'PRIMARY_FUEL_TYPE', 'SEGMENTATION', 'MAX_SHIP_WEIGHT', 'ORIGIN', 'CONFIG_COUNT']

# one row per cohort: specs are identical across a cohort's snapshots, so any row works;
# VIO_COUNT is dropped because it varies by snapshot
cohort_specs = (df_cohort
                .drop_duplicates(subset=['YEAR_MODEL', 'MAKE', 'MODEL'])
                [['YEAR_MODEL', 'MAKE', 'MODEL'] + spec_cols].copy())

# vehicle class separates distribution categories (car specs != truck specs), from VINtelligence vehicle type
vint_class = {'P': 'Car', 'T': 'Truck'}
cohort_specs['VEHICLE_CLASS'] = cohort_specs['PRIMARY_VEH_TYPE'].map(vint_class)

# check: should be 0 (every cohort maps to Car or Truck)
# cohort_specs['VEHICLE_CLASS'].isna().sum()

In [ ]:
# fixed class order and colors, reused in every chart so each class keeps the same color
class_order = ['Car', 'Truck']
class_colors = {'Car': '#2a78d6', 'Truck': '#eb6834'}

# one row per cohort with its peak VIO and vehicle class; validate raises if the merge duplicates a cohort
cohort_summary = (df_cohort.groupby(['YEAR_MODEL', 'MAKE', 'MODEL'])['VIO_COUNT'].max().reset_index()
                  .merge(cohort_specs[['YEAR_MODEL', 'MAKE', 'MODEL', 'VEHICLE_CLASS']],
                         on=['YEAR_MODEL', 'MAKE', 'MODEL'], how='left', validate='one_to_one'))

In [ ]:
edges = 10 ** np.histogram_bin_edges(np.log10(cohort_summary['VIO_COUNT']), bins='fd')

fig, axes = plt.subplots(1, len(class_order), figsize=(5 * len(class_order), 4), sharex=True)
for ax, cls in zip(axes, class_order):
    vals = cohort_summary.loc[cohort_summary['VEHICLE_CLASS'] == cls, 'VIO_COUNT']
    ax.hist(vals, bins=edges, color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_xscale('log')
    ax.set_title(f'{cls} ({len(vals):,} cohorts, median {vals.median():,.0f})')
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Number of Cohorts')
fig.supxlabel('Peak VIO Count per Cohort (log scale)')
fig.suptitle('Distribution of Peak VIO Count by Vehicle Class')
plt.tight_layout()
plt.show()

**Observations -**

- Trucks have larger cohorts than cars: median peak VIO is \~13.4k for trucks versus \~2.0k for cars, reflecting fewer truck models, each with higher registration volume (e.g. full-size pickups and SUVs).
- Both classes range from single-vehicle cohorts to several hundred thousand vehicles, but cars have more small cohorts (10th percentile of 74 versus 311 for trucks), while most truck cohorts (71%) fall between 1k and 100k, compared with 58% of car cohorts.

**Thoughts -**

- Attrition must be measured as a percentage, since cohort sizes span several orders of magnitude.
- Small cohorts produce unstable attrition rates (losing 1 of 5 vehicles is 20%), so a minimum size threshold or weighting, set per class, is needed before modeling.

In [ ]:
year_class = (cohort_summary.groupby(['YEAR_MODEL', 'VEHICLE_CLASS']).size()
              .unstack(fill_value=0)[class_order])

fig, ax = plt.subplots(figsize=(12, 5))
bottom = np.zeros(len(year_class))
for cls in class_order:
    ax.bar(year_class.index, year_class[cls], bottom=bottom, width=1.0,
           color=class_colors[cls], edgecolor='white', linewidth=0.5, label=cls)
    bottom += year_class[cls].values

ax.set_xlabel('Model Year')
ax.set_ylabel('Number of Cohorts (Make + Model)')
ax.set_title('Distribution of Cohorts by Model Year and Vehicle Class')
ax.legend(frameon=False, loc='upper left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

**Observations -**

- Coverage begins at model year 1981, the first year with VINtelligence specifications.
- The number of car models peaks around 2017 and declines afterward, while truck models grow through most of the period; trucks outnumber cars in every model year from 2022 onward, consistent with the industry shift toward pickups and SUVs.
- Model years 2026 and 2027 are incomplete because those vehicles are still being sold into the market.

**Thoughts -**

- Pre-1981 cohorts are excluded because they have no specifications, so the oldest cohorts in scope are from the early 1980s, now about 45 years old.
- Model years still being sold must be excluded from attrition until their registrations stop growing.
- As trucks and SUVs make up a growing share of models, future collector candidates may increasingly come from these segments, which supports benchmarking attrition within each class.

In [ ]:
msrp_all = cohort_specs['BASE_MSRP'].dropna()
edges = 10 ** np.histogram_bin_edges(np.log10(msrp_all), bins='fd')

fig, axes = plt.subplots(1, len(class_order), figsize=(5 * len(class_order), 4), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['BASE_MSRP'].dropna()
    missing = cls_rows['BASE_MSRP'].isna().mean()
    ax.hist(vals, bins=edges, color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.set_title(f'{cls} ({len(vals):,} cohorts, median ${vals.median():,.0f})\n{missing:.1%} missing MSRP',
                 fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Number of Cohorts')
fig.supxlabel('Base MSRP (log scale, nominal dollars)')
fig.suptitle('Distribution of Base MSRP by Vehicle Class')
plt.tight_layout()
plt.show()

# cohort_specs.nlargest(10, 'BASE_MSRP')[['YEAR_MODEL', 'MAKE', 'MODEL', 'VEHICLE_CLASS', 'BASE_MSRP']]
# cohort_specs.groupby('VEHICLE_CLASS')['BASE_MSRP'].describe(percentiles=[0.10, 0.50, 0.90, 0.99]).loc[class_order]

**Observations -**

- Cars and trucks share a similar median MSRP (\~$26.6k), but cars have a much longer high-price tail (99th percentile \~$415k versus \~$130k for trucks), including a distinct exotic and luxury cluster around $200k to $500k.
- The highest-priced cars are all limited-production hypercars (e.g. Bugatti Chiron, Ferrari F60 America), confirming the tail reflects real vehicles.
- MSRP is missing for \~5% of car cohorts and \~7% of truck cohorts.

**Thoughts -**

- Higher relative price is a candidate enthusiast indicator.
- Hypercars have very small populations and are collectible by design, so emerging collectability is more likely to appear among mainstream vehicles.
- MSRP is in nominal dollars; a within-model-year price rank will be used to remove inflation.

In [ ]:
disp_all = cohort_specs['MAX_ENGINE_DISP'].dropna()
edges = np.histogram_bin_edges(disp_all, bins='fd')

fig, axes = plt.subplots(1, len(class_order), figsize=(5 * len(class_order), 4), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['MAX_ENGINE_DISP'].dropna()
    missing = cls_rows['MAX_ENGINE_DISP'].isna().mean()
    ax.hist(vals, bins=edges, color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.set_title(f'{cls} ({len(vals):,} cohorts, median {vals.median():,.0f} ci)\n{missing:.1%} missing displacement',
                 fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Number of Cohorts')
fig.supxlabel('Max Engine Displacement (cubic inches)')
fig.suptitle('Distribution of Max Engine Displacement by Vehicle Class')
plt.tight_layout()
plt.show()

# cohort_specs.groupby('VEHICLE_CLASS')['MAX_ENGINE_DISP'].describe(percentiles=[0.10, 0.50, 0.90, 0.99]).loc[class_order]

**Observations -**

- Median max displacement is \~183 ci (\~3.0L) for cars and \~230 ci (\~3.8L) for trucks; trucks run larger at every percentile.
- Displacement clusters at common engine sizes rather than forming a smooth curve, reflecting shared engine families (four-cylinder, V6, V8) reused across many models. Trucks show a distinct big-block cluster near \~455 ci (\~7.4L to 7.5L).
- The car maximum of 512 ci (8.4L) matches the Dodge Viper V10, supporting data validity.

**Thoughts -**

- Large displacement, particularly in cars, is a candidate performance and enthusiast indicator.
- Because displacement groups into engine families, it is likely strongly correlated with cylinder count; the correlation analysis will determine whether both are needed as model features.
- Electric vehicles have no displacement, so their missing values must be treated as "no engine," not "unknown"; electric vehicles account for 482 of the 513 cohorts missing displacement.

In [ ]:
# cylinder counts are discrete, so bars per value; shares make car and truck panels comparable
# 0 = electric; rotary and unknown cylinder codes are recorded as missing
cyl_values = sorted(cohort_specs['MAX_CYLINDERS'].dropna().unique())
cyl_labels = [f'{v:g}' for v in cyl_values]

fig, axes = plt.subplots(1, len(class_order), figsize=(5 * len(class_order), 4), sharey=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['MAX_CYLINDERS'].dropna()
    missing = cls_rows['MAX_CYLINDERS'].isna().mean()
    shares = vals.value_counts(normalize=True).reindex(cyl_values, fill_value=0) * 100
    ax.bar(cyl_labels, shares.values, color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_title(f'{cls} ({len(vals):,} cohorts)\n{missing:.1%} missing cylinder count', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Share of Cohorts (%)')
fig.supxlabel('Max Cylinder Count (0 = electric)')
fig.suptitle('Distribution of Max Cylinder Count by Vehicle Class')
plt.tight_layout()
plt.show()

# cohort counts per cylinder value, by class
# pd.crosstab(cohort_specs['MAX_CYLINDERS'], cohort_specs['VEHICLE_CLASS'])[class_order]

**Observations -**

- Cars are led by four-cylinder engines (\~35%), followed by V6 (\~31%) and V8 (\~24%); trucks skew larger, led by V8 (\~37%) and V6 (\~36%).
- V10 engines are more common in trucks (\~1.6%) than cars (\~0.9%), consistent with heavy-duty pickup engines.
- \~5% of car cohorts offer a V12. This reflects two effects: the maximum available engine is recorded (e.g. a V12 top trim counts for the whole model), and low-volume exotic makers contribute many small cohorts.
- Electric vehicles are mostly recorded with a missing cylinder count rather than 0; fuel type identifies them more reliably (3.3% of cars, 3.8% of trucks).

**Thoughts -**

- High cylinder counts (V10, V12, W16) are strong performance indicators, but because they reflect the top available engine rather than the typical one, they identify models that offered high performance rather than vehicles that were typically high performance.
- Cohort shares describe the variety of models, not the vehicle population; weighting by registrations would show a much smaller V12 share.

In [ ]:
# manual share among cohorts with known transmission data, by model year
min_known = 30   # hide years where the share rests on too few cohorts
grp = cohort_specs.groupby(['YEAR_MODEL', 'VEHICLE_CLASS'])['HAS_MANUAL']
manual_by_year = (grp.mean() * 100).unstack()[class_order]
known_by_year = grp.count().unstack()[class_order]
manual_by_year = manual_by_year.where(known_by_year >= min_known).loc[1981:2025]

fig, ax = plt.subplots(figsize=(10, 4))
for cls in class_order:
    ax.plot(manual_by_year.index, manual_by_year[cls], color=class_colors[cls], linewidth=2, label=cls)
ax.set_ylim(0, 100)
ax.set_xlabel('Model Year')
ax.set_ylabel('Manual Offered (%)')
ax.set_title(f'Manual Transmission Availability by Model Year (years with {min_known}+ known cohorts)')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

**Observations -**

- Manual transmission availability declines over time in both classes. Cars start near 70% of models in the late 1980s and early 1990s, hold between 56% and 61% from 1996 through 2007, then fall to \~11% to 19% in the 2020s.
- Trucks start higher (90% or more in the late 1980s) and fall faster, dropping below cars by 1991 and to \~2% by 2025, with a sharp drop between 2018 and 2019 (13% to 4%).
- Model years 1981 to 1984 are not shown because most models from those years have no transmission data in VINtelligence.

**Thoughts -**

- Because manual availability depends strongly on era, the flag alone overstates the enthusiast signal for older vehicles and understates it for recent ones. A manual offered in a 2020 model is far more distinctive than one in a 1990 model.
- Feature engineering should compare manual availability against the model's own era (e.g. relative to the share of models offering a manual in the same model year), the same approach planned for MSRP.

In [ ]:
# VINtelligence stores body style as a code; use the most common description for each code as its label
body_lookup = session.sql("""
    SELECT BODY_STYLE_CD AS CODE, MODE(BODY_STYLE_DESC) AS LABEL
    FROM WORKSPACE.ENGINEER.VINTELLIGENCE_RAW
    WHERE BODY_STYLE_CD IS NOT NULL
    GROUP BY 1
""").to_pandas()
body_labels = dict(zip(body_lookup['CODE'], body_lookup['LABEL'].str.title()))

cohort_specs['BODY_STYLE_NAME'] = cohort_specs['PRIMARY_BODY_STYLE'].map(body_labels)

# top 10 body styles per class; the rest grouped as Other
top_n = 10
fig, axes = plt.subplots(1, len(class_order), figsize=(6 * len(class_order), 4.5), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['BODY_STYLE_NAME'].dropna()
    missing = cls_rows['PRIMARY_BODY_STYLE'].isna().mean()
    shares = vals.value_counts(normalize=True) * 100
    if len(shares) > top_n:
        shares = pd.concat([shares.head(top_n), pd.Series({'Other': shares.iloc[top_n:].sum()})])
    ax.barh(shares.index[::-1], shares.values[::-1],
            color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Share of Cohorts (%)')
    ax.set_title(f'{cls} ({len(vals):,} cohorts)\n{missing:.1%} missing body style', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Primary Body Style by Vehicle Class (Top 10)')
plt.tight_layout()
plt.show()

# # cohort counts per body style, by class
# pd.crosstab(cohort_specs['BODY_STYLE_NAME'], cohort_specs['VEHICLE_CLASS'])[class_order] \
#   .sort_values(class_order[0], ascending=False)

**Observations -**

- Sedans make up half of car cohorts (4,270; 50.4%), followed by coupes (1,680; 19.8%), hatchbacks (1,287; 15.2%), convertibles (774; 9.1%), and wagons (423; 5.0%); hearses, limousines, and commercial chassis make up the remaining 0.4%.
- Coupes and convertibles, the classic enthusiast body styles, together account for 2,454 car cohorts (29.0%).
- The truck class is dominated by SUVs (3,390; 57.8%), followed by pickups (974; 16.6%) and passenger and cargo vans (921; 15.7%).
- 529 truck cohorts (9.0%) have commercial body styles (straight truck, cab chassis, cutaway, strip chassis, step van, motor home).
- 15 truck cohorts are convertibles, all small open-top SUVs from 1981 to 1993 (Suzuki Samurai, Sidekick, and Jimny).
- Body style is the most common style across a model's VIN patterns, so secondary styles such as a convertible version of a coupe are undercounted.

**Thoughts -**

- Coupe and convertible body styles are candidate enthusiast indicators for cars.
- Commercial body styles provide a way to separate commercial vehicles from light-duty trucks and SUVs, which is likely needed because commercial vehicles follow different usage and attrition patterns.
- A "convertible offered" flag (`HAS_CONVERTIBLE`) and the VIO convertible share (`SHARE_CONVERTIBLE`) now capture convertible versions that are not the most common style (see Update Since Milestone 2 at the end of this notebook).

In [ ]:
# VINtelligence stores market segment as a code; use the most common description for each code as its label
seg_lookup = session.sql("""
    SELECT SEGMENTATION_CD AS CODE, MODE(SEGMENTATION_DESC) AS LABEL
    FROM WORKSPACE.ENGINEER.VINTELLIGENCE_RAW
    WHERE SEGMENTATION_CD IS NOT NULL
    GROUP BY 1
""").to_pandas()
seg_labels = dict(zip(seg_lookup['CODE'], seg_lookup['LABEL']))

cohort_specs['SEGMENT_NAME'] = cohort_specs['SEGMENTATION'].map(seg_labels)

# top 10 segments per class; the rest grouped as Other
top_n = 10
fig, axes = plt.subplots(1, len(class_order), figsize=(6.5 * len(class_order), 4.5), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['SEGMENT_NAME'].dropna()
    missing = cls_rows['SEGMENTATION'].isna().mean()
    shares = vals.value_counts(normalize=True) * 100
    if len(shares) > top_n:
        shares = pd.concat([shares.head(top_n), pd.Series({'Other': shares.iloc[top_n:].sum()})])
    ax.barh(shares.index[::-1], shares.values[::-1],
            color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Share of Cohorts (%)')
    ax.set_title(f'{cls} ({len(vals):,} cohorts)\n{missing:.1%} missing segment', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Market Segment by Vehicle Class (Top 10)')
plt.tight_layout()
plt.show()

# # cohort counts per segment, by class (reset_index keeps segment names in exports)
# pd.crosstab(cohort_specs['SEGMENT_NAME'], cohort_specs['VEHICLE_CLASS'])[class_order] \
#   .sort_values(class_order[0], ascending=False).reset_index()

**Observations -**

- Car segments group into three families: mainstream (41.3%), luxury (34.7%), and sport (23.1%, across Luxury Sport, Compact Sport, Prestige Sport, and Mid Size Sport).
- Truck segments are dominated by utility (SUV) segments (57.8%, including 21.7% luxury utility), followed by pickups (18.8%), vans (17.0%), and the Commercial Truck segment (6.4%).
- Utility segment counts (3,341) closely match SUV body style counts (3,390), supporting consistency between the two fields.
- Segment is missing for only 2.8% of car cohorts and 1.6% of truck cohorts.

**Thoughts -**

- Sport segments are the most direct enthusiast indicator available, since they are an explicit classification rather than a proxy such as price or displacement.
- Grouping segments into families (sport, luxury, mainstream for cars; utility, pickup, van, commercial for trucks) makes segment usable as a model feature and resolves truck fragmentation.
- The Commercial Truck segment, alongside commercial body styles, supports separating commercial vehicles from light-duty trucks and SUVs.

In [ ]:
# VINtelligence stores drive type as a code; use the most common description for each code as its label
drive_lookup = session.sql("""
    SELECT DRV_TYP_CD AS CODE, MODE(DRV_TYP_DESC) AS LABEL
    FROM WORKSPACE.ENGINEER.VINTELLIGENCE_RAW
    WHERE DRV_TYP_CD IS NOT NULL
    GROUP BY 1
""").to_pandas()
drive_labels = dict(zip(drive_lookup['CODE'], drive_lookup['LABEL']))

cohort_specs['DRIVE_TYPE_NAME'] = cohort_specs['PRIMARY_DRIVE_TYPE'].map(drive_labels)

# fixed order so each drive type sits in the same row in both panels
drive_order = [drive_labels[c] for c in ['RWD', '4RD', 'FWD', '4FD', 'AWD', 'U'] if c in drive_labels]

fig, axes = plt.subplots(1, len(class_order), figsize=(6 * len(class_order), 4), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['DRIVE_TYPE_NAME'].dropna()
    missing = cls_rows['PRIMARY_DRIVE_TYPE'].isna().mean()
    shares = vals.value_counts(normalize=True).reindex(drive_order, fill_value=0) * 100
    ax.barh(shares.index[::-1], shares.values[::-1],
            color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Share of Cohorts (%)')
    ax.set_title(f'{cls} ({len(vals):,} cohorts)\n{missing:.1%} missing drive type', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Primary Drive Type by Vehicle Class')
plt.tight_layout()
plt.show()

# # cohort counts per drive type, by class
# pd.crosstab(cohort_specs['DRIVE_TYPE_NAME'], cohort_specs['VEHICLE_CLASS'])[class_order] \
#   .reindex(drive_order).reset_index()

**Observations -**

- Front-wheel drive is the most common primary drive type for cars (3,859; 45.6%), followed by rear-wheel drive (3,311; 39.1%) and all-wheel drive (1,297; 15.3%). No car cohorts are primarily 4x4.
- Trucks split across rear-wheel drive (1,856; 31.6%), all-wheel drive (1,814; 30.9%), rear-wheel drive with 4x4 (1,099; 18.7%), and front-wheel drive (1,014; 17.3%), with front-wheel drive with 4x4 rare (85; 1.4%).
- Half of truck cohorts (2,955; 50.4%) are rear-drive based (rear-wheel drive or rear-wheel drive with 4x4), reflecting pickups and body-on-frame SUVs; the remainder are largely front- and all-wheel-drive crossovers and vans.
- Primary rear-wheel drive shares are lower than the "rear-wheel drive offered" flag (cars 39.1% vs 49.5%, trucks 31.6% vs 49.8%), because many models were offered in more than one drive type.
- Drive type is missing for only 4 cohorts, and no cohort is primarily "Unknown."

**Thoughts -**

- Rear-wheel drive is a meaningful enthusiast indicator for cars because it is not the default: front-wheel drive is more common.
- The primary drive type and the drive type availability flags (`HAS_RWD`, plus the new `HAS_FWD`, `HAS_AWD`, and `HAS_4WD`) answer different questions (typical configuration versus availability), so both may be useful as features. The flags are more reliable for the \~9% of cohorts whose VIN patterns tie between two or more drive types.
- Like manual transmissions, rear-wheel drive availability in cars likely shifted over time as front-wheel drive became standard; interpreting it relative to model year may strengthen the signal.

In [ ]:
# VINtelligence stores fuel type as a code; use the most common description for each code as its label
fuel_lookup = session.sql("""
    SELECT ENG_FUEL_CD AS CODE, MODE(ENG_FUEL_DESC) AS LABEL
    FROM WORKSPACE.ENGINEER.VINTELLIGENCE_RAW
    WHERE ENG_FUEL_CD IS NOT NULL
    GROUP BY 1
""").to_pandas()
fuel_labels = dict(zip(fuel_lookup['CODE'], fuel_lookup['LABEL']))

cohort_specs['FUEL_TYPE_NAME'] = cohort_specs['PRIMARY_FUEL_TYPE'].map(fuel_labels)

# fixed order (most common overall first) so each fuel sits in the same row in both panels
fuel_order = cohort_specs['FUEL_TYPE_NAME'].value_counts().index.tolist()

fig, axes = plt.subplots(1, len(class_order), figsize=(6 * len(class_order), 4), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['FUEL_TYPE_NAME'].dropna()
    missing = cls_rows['PRIMARY_FUEL_TYPE'].isna().mean()
    shares = vals.value_counts(normalize=True).reindex(fuel_order, fill_value=0) * 100
    bars = ax.barh(shares.index[::-1], shares.values[::-1],
                   color=class_colors[cls], edgecolor='white', linewidth=0.5)
    # labels make small fuels readable next to the dominant gasoline bar
    ax.bar_label(bars, labels=[f'{v:.1f}%' for v in shares.values[::-1]],
                 fontsize=8, color='#52514e', padding=2)
    ax.set_xlabel('Share of Cohorts (%)')
    ax.set_title(f'{cls} ({len(vals):,} cohorts)\n{missing:.1%} missing fuel type', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Primary Fuel Type by Vehicle Class')
plt.tight_layout()
plt.show()

# cohort counts per fuel type, by class
# pd.crosstab(cohort_specs['FUEL_TYPE_NAME'], cohort_specs['VEHICLE_CLASS'])[class_order] \
#   .reindex(fuel_order).reset_index()

**Observations -**

- Gasoline dominates both classes (cars 91.6%, trucks 84.0%).
- Diesel is far more common in trucks (6.1%) than cars (0.5%), consistent with heavy-duty pickups and commercial vehicles; flex-fuel is also more common in trucks (4.2% vs 1.1%).
- Electric vehicles account for 3.3% of car cohorts and 3.8% of truck cohorts; hybrids account for 3.2% and 1.7%.
- Electric vehicles account for 482 of the 513 cohorts missing engine displacement, indicating that missing displacement for these vehicles reflects having no engine rather than missing data.
- Fuel type is complete for all cohorts.
- Primary fuel type is the most common fuel across a model's VIN patterns, so hybrid and flex-fuel versions of mainly gasoline models are undercounted.

**Thoughts -**

- Electric vehicles need explicit handling in engine-based features: missing displacement and cylinder counts for electric vehicles should be treated as "no engine," and fuel type can be used to identify them.
- Electric and hybrid cohorts are concentrated in recent model years, so they will have limited attrition history within the data window.

In [ ]:
# VINtelligence stores origin as a code; use the most common description for each code as its label
# (MODE also resolves inconsistent casing in the source, e.g. 'DOMESTIC' vs 'Domestic')
origin_lookup = session.sql("""
    SELECT ORGN_CD AS CODE, MODE(ORGN_DESC) AS LABEL
    FROM WORKSPACE.ENGINEER.VINTELLIGENCE_RAW
    WHERE ORGN_CD IS NOT NULL
    GROUP BY 1
""").to_pandas()
origin_labels = dict(zip(origin_lookup['CODE'], origin_lookup['LABEL']))

cohort_specs['ORIGIN_NAME'] = cohort_specs['ORIGIN'].map(origin_labels)

# logical order from domestic to import, so both panels read the same way
origin_order = [origin_labels[c] for c in ['D', 'Z', 'B', 'I'] if c in origin_labels]

fig, axes = plt.subplots(1, len(class_order), figsize=(6 * len(class_order), 3.5), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['ORIGIN_NAME'].dropna()
    missing = cls_rows['ORIGIN'].isna().mean()
    shares = vals.value_counts(normalize=True).reindex(origin_order, fill_value=0) * 100
    ax.barh(shares.index[::-1], shares.values[::-1],
            color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Share of Cohorts (%)')
    ax.set_title(f'{cls} ({len(vals):,} cohorts)\n{missing:.1%} missing origin', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Vehicle Origin by Vehicle Class')
plt.tight_layout()
plt.show()

# # cohort counts per origin, by class
# pd.crosstab(cohort_specs['ORIGIN_NAME'], cohort_specs['VEHICLE_CLASS'])[class_order] \
#   .reindex(origin_order).reset_index()

**Observations -**

- Most car cohorts are imports (5,056; 60.4%), with domestic cars at 28.2%; trucks lean domestic (2,788; 47.9%), with imports at 31.0%.
- Foreign-brand vehicles built in North America are twice as common among trucks (17.8%) as cars (8.9%), reflecting the North American production of foreign-brand SUVs and pickups.
- US brands account for 30.7% of car cohorts but 51.2% of truck cohorts, consistent with US brands' long-standing strength in pickups and full-size SUVs.
- Shares count models, not vehicles; foreign brands offer many low-volume models, so imports are over-represented relative to registered vehicles.
- Origin is missing for \~1% of cohorts (101 cars, 45 trucks).

**Thoughts -**

- Origin groups very different vehicles under "Import" (e.g. Japanese, German, Italian). The new assembly plant country column (`PRIMARY_PLANT_COUNTRY`) shows where vehicles were built, but a brand country derived from make would still be needed for collectability questions such as collector interest in 1990s Japanese vehicles.
- Weighting by registrations would give a more realistic picture of the vehicle population than cohort counts.

In [ ]:
weight_all = cohort_specs['MAX_SHIP_WEIGHT'].dropna()
edges = np.histogram_bin_edges(weight_all, bins='fd')

fig, axes = plt.subplots(1, len(class_order), figsize=(5 * len(class_order), 4), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['MAX_SHIP_WEIGHT'].dropna()
    missing = cls_rows['MAX_SHIP_WEIGHT'].isna().mean()
    ax.hist(vals, bins=edges, color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.set_title(f'{cls} ({len(vals):,} cohorts, median {vals.median():,.0f} lbs)\n{missing:.1%} missing weight',
                 fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Number of Cohorts')
fig.supxlabel('Max Shipping Weight (lbs)')
fig.suptitle('Distribution of Max Shipping Weight by Vehicle Class')
plt.tight_layout()
plt.show()

# cohort_specs.nsmallest(5, 'MAX_SHIP_WEIGHT')[['YEAR_MODEL', 'MAKE', 'MODEL', 'VEHICLE_CLASS', 'MAX_SHIP_WEIGHT']]
# cohort_specs[cohort_specs['VEHICLE_CLASS'] == 'Car'].nlargest(5, 'MAX_SHIP_WEIGHT')[['YEAR_MODEL', 'MAKE', 'MODEL', 'MAX_SHIP_WEIGHT']]
# cohort_specs.groupby('VEHICLE_CLASS')['MAX_SHIP_WEIGHT'].describe(percentiles=[0.10, 0.50, 0.90, 0.99]).loc[class_order]

**Observations -**

- Median shipping weight is 3,450 lbs for cars and 4,456 lbs for trucks; 80% of cars fall between \~2,400 and 4,400 lbs, and 80% of trucks between \~3,300 and 6,100 lbs.
- Cars form a single peak around 3,500 lbs, while trucks show a second cluster around 5,500 to 6,000 lbs (heavy-duty pickups, full-size vans, and large SUVs) and a thin commercial tail up to \~9,700 lbs.
- Extremes are plausible except one: the lightest cohorts are low-speed electric vehicles (\~1,135 to 1,140 lbs) and the heaviest trucks are heavy-duty pickups and vans, but the heaviest car (2011 Lexus LS, 7,420 lbs) is a source data error from a single VIN pattern.
- Weight is the least complete numeric specification, missing for 570 cars (6.7%) and 598 trucks (10.2%); only the manual transmission flag is missing more often (\~18%).

**Thoughts -**

- Displacement relative to weight is a simple power-to-weight proxy that may separate performance vehicles from ordinary ones better than either measure alone. VINtelligence's own power to weight ratio (`MIN_LBS_PER_HP`) is only available from model year 2000 onward, so this proxy remains the option for older cohorts.
- Maximum-based specifications reflect the heaviest or most capable version of a model (e.g. heavy-duty Silverado) and are sensitive to a single erroneous VIN pattern, so outliers should be checked before modeling.

In [ ]:
# door counts are discrete, so bars per value; shares make car and truck panels comparable
door_values = sorted(cohort_specs['MAX_DOOR_CNT'].dropna().unique())
door_labels = [f'{v:g}' for v in door_values]

fig, axes = plt.subplots(1, len(class_order), figsize=(5 * len(class_order), 4), sharey=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['MAX_DOOR_CNT'].dropna()
    missing = cls_rows['MAX_DOOR_CNT'].isna().mean()
    shares = vals.value_counts(normalize=True).reindex(door_values, fill_value=0) * 100
    ax.bar(door_labels, shares.values, color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_title(f'{cls} ({len(vals):,} cohorts)\n{missing:.1%} missing door count', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Share of Cohorts (%)')
fig.supxlabel('Max Door Count')
fig.suptitle('Distribution of Max Door Count by Vehicle Class')
plt.tight_layout()
plt.show()

# cohort counts per door count, by class
# pd.crosstab(cohort_specs['MAX_DOOR_CNT'], cohort_specs['VEHICLE_CLASS'])[class_order].reset_index()

**Observations -**

- Most cohorts have a maximum of 4 doors (cars 68.3%, trucks 74.8%); 2-door cohorts make up 30.9% of cars and 15.1% of trucks.
- 3-door configurations appear mainly in trucks (9.7%), likely extended-cab pickups and minivans with a single sliding door; cars rarely have odd door counts, so hatchbacks are recorded by passenger doors.
- The 2-door car share (30.9%) closely matches the coupe and convertible body style share (29.0%), showing that door count largely duplicates body style.
- Door count is complete for cars; 226 truck cohorts (3.9%) lack it.

**Thoughts -**

- Door count adds little beyond body style and is a candidate for removal during feature selection; the correlation analysis should confirm this.

In [ ]:
# number of distinct configurations per cohort (trim, body style, displacement, drive type, transmission)
config_all = cohort_specs['CONFIG_COUNT'].dropna()
edges = 10 ** np.histogram_bin_edges(np.log10(config_all), bins='fd')

fig, axes = plt.subplots(1, len(class_order), figsize=(5 * len(class_order), 4), sharex=True)
for ax, cls in zip(axes, class_order):
    cls_rows = cohort_specs[cohort_specs['VEHICLE_CLASS'] == cls]
    vals = cls_rows['CONFIG_COUNT'].dropna()
    missing = cls_rows['CONFIG_COUNT'].isna().mean()
    ax.hist(vals, bins=edges, color=class_colors[cls], edgecolor='white', linewidth=0.5)
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.set_title(f'{cls} ({len(vals):,} cohorts, median {vals.median():,.0f} configurations)\n{missing:.1%} missing',
                 fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Number of Cohorts')
fig.supxlabel('Configurations per Cohort (log scale)')
fig.suptitle('Distribution of Configuration Count by Vehicle Class')
plt.tight_layout()
plt.show()

# cohort_specs.groupby('VEHICLE_CLASS')['CONFIG_COUNT'].describe(percentiles=[0.10, 0.50, 0.90, 0.99]).loc[class_order]

**Observations -**

- Trucks are offered in more configurations than cars: median configuration count is 5 for trucks versus 3 for cars, and the 99th percentile is 59 versus 24.
- About a quarter of car cohorts have a single configuration, indicating models offered in only one trim, engine, and body combination.
- Configuration count replaced VIN pattern count, which was distorted by manufacturers encoding different levels of detail in their VINs (e.g. 2015 Volvo V60: 1,032 VIN patterns but 26 configurations) and by some models storing individual vehicle VINs.
- Configuration count is complete for all cohorts.

**Thoughts -**

- Configuration count measures variety, not production volume; it may act as a proxy for production scale if it correlates with peak VIO, which the correlation analysis will test.
- The right skew suggests a log transformation before using it as a model feature.

In [ ]:
# 0/1 indicators for categories tied to the hypotheses; NaN stays NaN so a missing value is never counted as "no"
def indicator(col, values):
    return cohort_specs[col].isin(values).astype(float).where(cohort_specs[col].notna())

sport_segments = ['Luxury Sport', 'Compact Sport', 'Prestige Sport', 'Mid Size Sport']
luxury_segments = ['Mid Size Luxury Car', 'Compact Luxury Car', 'Full Size Luxury Car', 'Subcompact Luxury Car',
                   'Full Size Luxury Prestige', 'Compact Luxury Utility', 'Subcompact Plus Luxury Utility',
                   'Lower Mid Size Luxury Utility', 'Upper Mid Size Luxury Utility', 'Full Size Luxury Utility',
                   'Full Size Luxury Prestige Utility', 'Subcompact Luxury Utility', 'Full Size Half Ton Luxury Pickup']
commercial_styles = ['Straight Truck', 'Cab Chassis', 'Cutaway', 'Incomplete (Strip Chassis)', 'Step Van', 'Motor Home']

cohort_specs['IS_SPORT_SEGMENT']  = indicator('SEGMENT_NAME', sport_segments)
cohort_specs['IS_LUXURY_SEGMENT'] = indicator('SEGMENT_NAME', luxury_segments)
cohort_specs['IS_COUPE_CONV']     = indicator('BODY_STYLE_NAME', ['Coupe', 'Convertible'])
cohort_specs['IS_COMMERCIAL']     = indicator('BODY_STYLE_NAME', commercial_styles)
cohort_specs['IS_FWD']            = indicator('DRIVE_TYPE_NAME', ['Front Wheel Drive'])
cohort_specs['IS_ELECTRIC']       = indicator('FUEL_TYPE_NAME', ['Electric'])
cohort_specs['IS_IMPORT']         = indicator('ORIGIN_NAME', ['Import', 'Import Built in North America'])

# sanity check: share of cohorts flagged, by class (NaN = source value missing)
cohort_specs.groupby('VEHICLE_CLASS')[[c for c in cohort_specs if c.startswith('IS_')]].mean().round(3).reindex(class_order)

In [ ]:
# Spearman rank correlation: suited to skewed, discrete, and 0/1 features, and robust to outliers
corr_cols = {'BASE_MSRP': 'MSRP', 'MAX_ENGINE_DISP': 'Displacement', 'MAX_CYLINDERS': 'Cylinders',
             'MAX_SHIP_WEIGHT': 'Weight', 'MAX_DOOR_CNT': 'Doors', 'HAS_MANUAL': 'Manual',
             'HAS_RWD': 'RWD', 'CONFIG_COUNT': 'Configurations', 'PEAK_VIO': 'Peak VIO',
             'YEAR_MODEL': 'Model Year',
             'IS_SPORT_SEGMENT': 'Sport Segment', 'IS_LUXURY_SEGMENT': 'Luxury Segment',
             'IS_COUPE_CONV': 'Coupe/Convertible', 'IS_COMMERCIAL': 'Commercial',
             'IS_FWD': 'Front-Wheel Drive', 'IS_ELECTRIC': 'Electric', 'IS_IMPORT': 'Import Brand'}

# one row per cohort with specs, plus each cohort's peak VIO
corr_data = (cohort_specs[cohort_specs['CONFIG_COUNT'].notna()]
             .merge(cohort_summary[['YEAR_MODEL', 'MAKE', 'MODEL', 'VIO_COUNT']],
                    on=['YEAR_MODEL', 'MAKE', 'MODEL'], how='left', validate='one_to_one')
             .rename(columns={'VIO_COUNT': 'PEAK_VIO'}))

# diverging scale: blue = negative, neutral gray = none, red = positive
div_cmap = LinearSegmentedColormap.from_list('div', ['#2a78d6', '#f0efec', '#e34948'])

top_pairs = {}
fig, axes = plt.subplots(1, len(class_order), figsize=(9 * len(class_order), 8))
for ax, cls in zip(axes, class_order):
    cls_data = corr_data.loc[corr_data['VEHICLE_CLASS'] == cls, list(corr_cols)].rename(columns=corr_cols)
    cls_data = cls_data.loc[:, cls_data.nunique() > 1]   # drop indicators with no variation in this class
    corr = cls_data.corr(method='spearman')
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)   # show each pair once
    sns.heatmap(corr, mask=mask, ax=ax, cmap=div_cmap, vmin=-1, vmax=1, center=0,
                annot=True, fmt='.2f', annot_kws={'size': 7}, square=True,
                linewidths=0.5, linecolor='white', cbar=(cls == class_order[-1]))
    ax.set_title(f'{cls} ({len(cls_data):,} cohorts)')

    # strongest pairs for the written observations
    lower = corr.where(np.tril(np.ones_like(corr, dtype=bool), k=-1)).stack()
    top_pairs[cls] = lower.reindex(lower.abs().sort_values(ascending=False).index).head(12).round(2)

fig.suptitle('Spearman Correlation of Vehicle Features by Class')
plt.tight_layout()
plt.show()

# 12 strongest correlations per class (positive or negative)
# pd.concat(top_pairs, axis=1)

# era check: manual vs MSRP after removing the model year effect
# (MSRP rank within model year vs manual availability relative to its model year)
for cls in class_order:
    d = corr_data[corr_data['VEHICLE_CLASS'] == cls].copy()
    d['MSRP_RANK_IN_YEAR'] = d.groupby('YEAR_MODEL')['BASE_MSRP'].rank(pct=True)
    d['MANUAL_VS_ERA'] = d['HAS_MANUAL'] - d.groupby('YEAR_MODEL')['HAS_MANUAL'].transform('mean')
    era_corr = d[['MSRP_RANK_IN_YEAR', 'MANUAL_VS_ERA']].corr(method='spearman').iloc[0, 1]
    print(f'{cls}: manual vs MSRP, adjusted for model year = {era_corr:.2f}')

**Observations -**

- Displacement and cylinder count are nearly redundant (0.94 cars, 0.91 trucks) and both correlate with weight (0.61 to 0.71), forming a single engine size and weight cluster.
- Door count largely duplicates body style and segment in cars (−0.79 with coupe/convertible, −0.76 with sport segment); in trucks it tracks model year (0.58) and MSRP (0.55) instead.
- Sport segment is only weakly correlated with MSRP, displacement, cylinders, manual, and RWD (0.15 to 0.27), so it adds information beyond price and engine size. It overlaps partly with coupe/convertible (0.66).
- Manual transmission and rear-wheel drive are essentially uncorrelated in cars (−0.08), so they are distinct signals.
- Model year is strongly related to MSRP (0.60 cars, 0.76 trucks) and negatively to manual availability (−0.35 cars, −0.54 trucks). Adjusting for model year only partly reduces the manual-MSRP correlation (cars −0.49 to −0.40, trucks −0.66 to −0.45), so manual availability remains associated with lower-priced vehicles within the same model year.
- Configuration count is moderately related to peak VIO (0.40 cars, 0.54 trucks), a partial proxy for production scale.
- The blank cell in the truck panel (weight vs coupe/convertible) occurs because none of the 15 truck convertibles has a shipping weight.

**Thoughts -**

- For modeling, keep one of displacement or cylinder count, and drop door count for cars.
- Keep sport segment, manual, and RWD as separate features; sport segment and coupe/convertible partly overlap.
- The era effect supports the planned within-model-year MSRP rank and era-relative manual availability, but manual availability is not a clean enthusiast signal, since it also marks lower-priced vehicles. Its value may depend on combination with other features (e.g. sport segment, RWD).
- Peak VIO is partly a survivorship measure for older cohorts, so it should not stand in for production volume without caution.

## Update Since Milestone 2

- **New specification columns.** `VIN_MODEL_AGG` now has 44 columns (previously 17), adding price, performance, body and roof, trim, truck, fuel economy, and market attributes, plus drive type availability flags (`HAS_FWD`, `HAS_AWD`, `HAS_4WD`) alongside `HAS_RWD`. Horsepower, torque, power to weight, engine location, seat count, height, fuel economy, towing, and payload are only populated from model year 2000 onward.
- **New registration shares.** `VIO_COHORT` now records, for each cohort and snapshot, the share of registrations that are convertibles, coupes, regular cab pickups, and work bodies (cargo and step vans, cab chassis, cutaways, straight trucks, and incomplete chassis), and the share registered for personal use. These come from VIO, so they count registered vehicles rather than VIN patterns.
- **Repeatable "most common" values.** `MODE` picks arbitrarily when two values tie (in scope, \~9% of cohorts tie on drive type and \~7% on body style), so values could change on every rebuild. Each "most common" value now uses a fixed, neutral tie break, so rebuilds give identical results and no category is favored.
- **Effect on this notebook.** Row counts, registration totals, and the 14,339 cohort scope are unchanged; `VIO_COHORT_ENRICHED` now has 53 columns. The resolved ties shift some body style, drive type, fuel type, origin, and segment counts slightly, and the figures and observations above reflect the updated tables. The new columns are evaluated in Feature Engineering and used in the models below.

## Data Quality Checks

Resolve data issues before building features. Values are standardized first, then likely errors are corrected, missing values are filled, and finally the cohorts are checked for sampling bias.

In [ ]:
# working copy for Milestone 3; df_cohort stays as loaded so the EDA above is unchanged
df_clean = df_cohort.copy()
cohort_keys = ['YEAR_MODEL', 'MAKE', 'MODEL']
df_clean['VEHICLE_CLASS'] = df_clean['PRIMARY_VEH_TYPE'].map(vint_class)

### Snapshot Coverage

Check that each cohort appears consistently across the 20 quarterly snapshots, and look for gaps or sudden jumps between snapshots.

In [ ]:
# cohorts and registered vehicles per snapshot, with the change in vehicles from the previous snapshot
coverage = df_clean.groupby('SNAPSHOT_DATE').agg(COHORTS=('MODEL', 'size'), VEHICLES=('VIO_COUNT', 'sum'))
coverage['VEHICLE_CHANGE_PCT'] = coverage['VEHICLES'].pct_change() * 100
coverage

In [ ]:
# cohorts missing from a snapshot between their first and last appearance
snap_rank = df_clean['SNAPSHOT_DATE'].rank(method='dense')
span = snap_rank.groupby([df_clean[k] for k in cohort_keys]).agg(['min', 'max', 'count'])
gaps = span[span['max'] - span['min'] + 1 > span['count']]
cohort_peak = df_clean.groupby(cohort_keys)['VIO_COUNT'].max()
print(f"cohorts with a gap: {len(gaps)} (largest peak VIO among them: {cohort_peak[gaps.index].max()})")

# established cohorts (age 3+, 100+ vehicles) that grow more than 5% between snapshots; once a model year
# has been sold its registrations should only fall, so a large jump points to records moved between models
q = df_clean.sort_values(cohort_keys + ['SNAPSHOT_DATE'])
prev = q.groupby(cohort_keys)['VIO_COUNT'].shift()
age = q.groupby(cohort_keys)['SNAPSHOT_DATE'].shift().dt.year - q['YEAR_MODEL']
growth = (q['VIO_COUNT'] / prev - 1) * 100
jumps = q.assign(PREV_COUNT=prev, GROWTH_PCT=growth)[(age >= 3) & (prev >= 100) & (growth > 5)]
print(f"{len(jumps)} jumps in {jumps.groupby(cohort_keys).ngroups} cohorts")
display(jumps.nlargest(5, 'GROWTH_PCT')[cohort_keys + ['SNAPSHOT_DATE', 'PREV_COUNT', 'VIO_COUNT', 'GROWTH_PCT']])

# the 2007 Subaru Legacy and Outback exchange registrations in April 2024, so neither has a consistent history
subaru_2007 = (df_clean['YEAR_MODEL'] == 2007) & (df_clean['MAKE'] == 'SUBARU') & df_clean['MODEL'].isin(['LEGACY', 'OUTBACK'])
df_clean = df_clean[~subaru_2007]

**Observations -**

- All 20 snapshots are present. Cohorts grow from 12,290 (April 2021) to 14,263 (July 2026) as new model years enter, and total registrations change by at most 1.3% between snapshots (the largest is across the nine month gap after April 2021), so no snapshot is missing a block of data.
- 26 cohorts skip a snapshot between their first and last appearance; all peak at 7 or fewer vehicles, so these are tiny cohorts dropping in and out rather than missing data.
- 41 jumps of more than 5% appear in 35 established cohorts. The 2007 Subaru Outback grew 404% in April 2024 while the 2007 Legacy fell 64%, a reclassification between models, so both cohorts are removed. The other jumps are at most 16%, mostly in hydrogen (Toyota Mirai), exotic, and classic cohorts, and are kept as real registration changes; they will show up as negative attrition.

### Non-Standard Value Formatting

Standardize codes, labels, and placeholder values (such as blanks, "U" for unknown, or zeros) so each value means the same thing everywhere. This comes first so that hidden missing values are counted correctly.

In [ ]:
# text keys are already uppercase with no stray spaces (counts of values that would change)
print({c: int((df_clean[c] != df_clean[c].str.strip().str.upper()).sum()) for c in ['MAKE', 'MODEL']})

# readable labels for the coded categories, using the lookups built in the EDA
code_labels = {'PRIMARY_BODY_STYLE': ('BODY_STYLE_NAME', body_labels), 'SEGMENTATION': ('SEGMENT_NAME', seg_labels),
               'PRIMARY_DRIVE_TYPE': ('DRIVE_TYPE_NAME', drive_labels), 'PRIMARY_FUEL_TYPE': ('FUEL_TYPE_NAME', fuel_labels),
               'ORIGIN': ('ORIGIN_NAME', origin_labels)}
for code_col, (name_col, labels) in code_labels.items():
    df_clean[name_col] = df_clean[code_col].map(labels)

# values measured in a different unit than the rest of the column are removed:
# fuel economy for electric and hydrogen vehicles is MPGe (energy equivalent), not MPG, and the GMC Hummer EV
# reports wheel torque (11,500 lb-ft) rather than motor torque
no_engine = df_clean['PRIMARY_FUEL_TYPE'].isin(['E', 'R'])
wheel_torque = df_clean['MAX_TORQUE'] > 4 * df_clean['MAX_HORSEPOWER']
for mask, col in [(no_engine, 'MAX_MPG_COMBINED'), (wheel_torque, 'MAX_TORQUE')]:
    print(f"{col}: {df_clean.loc[mask & df_clean[col].notna()].groupby(cohort_keys).ngroups} cohorts removed")
    df_clean.loc[mask, col] = np.nan

# vehicle type recorded by VIO versus VINtelligence; the analysis uses VINtelligence, which is fixed per cohort
pd.crosstab(df_clean['VIO_VEHICLE_TYPE'], df_clean['VEHICLE_CLASS'])

**Observations -**

- `MAKE` and `MODEL` are already consistent (uppercase, no stray spaces), and the upstream fixes (numbers stored as text, the RWD code, zero placeholders, plant country spelling) already put specification values in one format.
- Readable label columns are added for the coded categories (body style, segment, drive type, fuel type, origin), matching the EDA.
- Two columns mixed units. Fuel economy is MPGe for electric and hydrogen vehicles (372 cohorts; median 104 and 67, versus 22 for gasoline), so those values are removed rather than compared with MPG. The GMC Hummer EV's torque (8 cohorts) is wheel torque, more than 11 times its horsepower, so it is removed too. Hybrid values (median 36, up to 136) may include MPGe for rechargeable versions and are kept.
- VIO and VINtelligence disagree on vehicle type for 85 rows; the analysis uses the VINtelligence class, which is fixed for each cohort.

### Outliers

Find extreme or impossible values and decide whether to correct, cap, or keep them.

In [ ]:
# a value far from the same model's neighboring model years (up to two on each side, which must agree within 20%)
# is likely a source error: size and weight change gradually (flag beyond 1.4x), engine output can jump with a new
# powertrain (flag beyond 3x); flagged values are replaced with the neighbors' median
tolerance = {'MAX_SHIP_WEIGHT': 1.4, 'MIN_WHEELBASE_IN': 1.4, 'MIN_HEIGHT_IN': 1.4, 'MAX_HORSEPOWER': 3, 'MAX_TORQUE': 3}
cohorts = df_clean.drop_duplicates(cohort_keys).sort_values(cohort_keys).set_index(cohort_keys)
row_keys = pd.MultiIndex.from_frame(df_clean[cohort_keys])
corrections = []
for col, tol in tolerance.items():
    by_model = cohorts.groupby(['MAKE', 'MODEL'])[col]
    neighbors = pd.concat([by_model.shift(k) for k in (-2, -1, 1, 2)], axis=1)
    baseline = neighbors.median(axis=1)
    stable = (neighbors.notna().sum(axis=1) >= 2) & (neighbors.max(axis=1) / neighbors.min(axis=1) <= 1.2)
    ratio = cohorts[col] / baseline
    fixed = baseline[stable & ((ratio > tol) | (ratio < 1 / tol))]
    corrections += [(col, *k, cohorts.loc[k, col], v) for k, v in fixed.items()]
    replacement = fixed.reindex(row_keys).to_numpy()
    df_clean[col] = np.where(pd.isna(replacement), df_clean[col], replacement)

corrections = pd.DataFrame(corrections, columns=['COLUMN'] + cohort_keys + ['ORIGINAL', 'REPLACED_WITH'])
print(corrections['COLUMN'].value_counts().to_dict())
corrections

**Observations -**

- 35 values in 34 cohorts are corrected. Most are shipping weights (27), including the 2011 Lexus LS noted in the EDA (7,420 lbs, now 5,159) and five 1991 GMC models whose weights are 51% to 87% above their neighboring years.
- The rest are impossible dimensions or output, such as a 5.5 inch tall 2024 Tesla Model 3, a 166.7 inch wheelbase for the 2018 Toyota 86, a 2,024 hp 2024 Lexus UX (likely the model year entered as horsepower), and 38 hp for the 2020 BMW i3.
- Real extremes are kept, such as hypercar prices up to $5.4M and horsepower above 1,000 (e.g. Corvette, Lucid Air). Skewed values such as price, peak VIO, and configuration count are handled with ranks or log transforms in Feature Engineering.
- The rule only catches isolated errors: a value that repeats in consecutive years, such as the 60.6 inch wheelbase for the 2010 and 2011 Lotus Elise (90.6 before), is not caught. Attrition outliers are reviewed in Attrition Metrics, once attrition is calculated.

### Missing Values

Decide how to handle each missing value (fill it, label it as unknown, or exclude it) based on why it is missing; for example, electric vehicles have no engine displacement.

In [ ]:
num_cols = ['BASE_MSRP', 'MAX_MSRP', 'MAX_ENGINE_DISP', 'MAX_CYLINDERS', 'MAX_SHIP_WEIGHT', 'MAX_DOOR_CNT',
            'MAX_HORSEPOWER', 'MAX_TORQUE', 'MIN_LBS_PER_HP', 'MIN_SEATS', 'MIN_WHEELBASE_IN', 'MIN_HEIGHT_IN',
            'TRIM_COUNT', 'MAX_TOW_CAPACITY', 'MAX_PAYLOAD', 'MAX_MPG_COMBINED']
before = df_clean.drop_duplicates(cohort_keys)[num_cols].isna().mean() * 100

# no combustion engine (electric, hydrogen fuel cell): missing displacement and cylinders mean zero, not unknown
engine_cols = ['MAX_ENGINE_DISP', 'MAX_CYLINDERS']
df_clean.loc[no_engine, engine_cols] = df_clean.loc[no_engine, engine_cols].fillna(0)

# other numeric specifications: fill with the median of the same model year and vehicle class (one row per cohort);
# a group with no values at all (e.g. horsepower before 2000) stays missing, which the era split handles
medians = df_clean.drop_duplicates(cohort_keys).groupby(['YEAR_MODEL', 'VEHICLE_CLASS'])[num_cols].median()
group_medians = df_clean[['YEAR_MODEL', 'VEHICLE_CLASS']].join(medians, on=['YEAR_MODEL', 'VEHICLE_CLASS'])
df_clean[num_cols] = df_clean[num_cols].fillna(group_medians[num_cols])
df_clean.loc[no_engine, 'MAX_MPG_COMBINED'] = np.nan   # MPG does not apply without an engine

# categorical specifications: missing becomes an explicit 'Unknown' category;
# yes/no flags (e.g. HAS_MANUAL) stay missing and are encoded as unknown in One-Hot Encoding
cat_cols = ['PRIMARY_DRIVE_TYPE', 'DRIVE_TYPE_NAME', 'SEGMENTATION', 'SEGMENT_NAME', 'ORIGIN', 'ORIGIN_NAME',
            'PRIMARY_ENGINE_BLOCK', 'PRIMARY_PLANT_COUNTRY', 'PARENT_MFG']
df_clean[cat_cols] = df_clean[cat_cols].fillna('Unknown')

after = df_clean.drop_duplicates(cohort_keys)[num_cols].isna().mean() * 100
pd.DataFrame({'Missing Before (%)': before, 'Missing After (%)': after}).round(1)

**Observations -**

- The 513 cohorts with no combustion engine and no displacement (482 electric, 31 hydrogen) now show zero displacement, and 420 show zero cylinders; fuel economy stays missing for them because MPG does not apply.
- Price (6.0% missing), shipping weight (8.1%), wheelbase (4.8%), door count (1.6%), and trim count (23.6%) are filled with the median of the same model year and vehicle class, so each filled value reflects its own era and class. The 23 other missing cylinder counts (including rotary engines) are filled the same way.
- What remains missing is structural: horsepower, torque, power to weight, seats, height, and fuel economy have no values before model year 2000 (34.6% of cohorts), and towing and payload are mainly truck specifications. These stay missing for the era split.
- Categorical gaps become an explicit "Unknown" category (engine block 590 cohorts, segment 330, parent manufacturer 312, plant country 158, origin 146, drive type 4). Yes/no flags without data, such as `HAS_MANUAL` (17.6% of cohorts), stay missing so they are encoded as unknown rather than "no".

### Sampling Bias

Check how well the cohorts in scope represent the vehicle population, including small cohorts, model years before 1981, and older cohorts that contain only surviving vehicles.

In [ ]:
# cohorts and peak registrations by group: models sold only outside the US, small cohorts, and the rest
peak = df_clean.groupby(cohort_keys).agg(PEAK_VIO=('VIO_COUNT', 'max'), NON_US=('NON_US_MARKET_ONLY', 'max'))
group = np.select([peak['NON_US'] == 1, peak['PEAK_VIO'] < 100], ['Sold outside US only', 'Peak under 100'], 'Peak 100+')
sample = peak.groupby(group)['PEAK_VIO'].agg(COHORTS='size', PEAK_VIO='sum')
sample[['COHORTS_PCT', 'PEAK_VIO_PCT']] = sample / sample.sum() * 100
display(sample)

# ages observed by model year decade: each cohort is seen for at most five years of its life
age = df_clean['SNAPSHOT_DATE'].dt.year - df_clean['YEAR_MODEL']
display(age.groupby(df_clean['YEAR_MODEL'] // 10 * 10).agg(['min', 'max']).rename_axis('DECADE'))

# models sold only outside the US are removed; small cohorts stay and are excluded from attrition in Attrition Metrics
df_clean = df_clean[df_clean['NON_US_MARKET_ONLY'] != 1]
print(f"df_clean: {len(df_clean):,} rows, {df_clean.groupby(cohort_keys).ngroups:,} cohorts")

**Observations -**

- 330 cohorts (2.3%) are models sold only outside the US, such as the Hyundai Pony and the Asuna brand from Canada, with a median of 4 registered vehicles. Their US registrations are not representative of their market, so they are removed.
- 1,095 cohorts (7.6%) peak below 100 vehicles but hold only 0.013% of peak registrations, led by Ferrari (77 cohorts), Rolls-Royce (72), and Bentley (65). They stay in the data, but annual periods starting below 100 vehicles are excluded in Attrition Metrics, so exotic vehicles that are collectible by design will be underrepresented in the attrition models.
- Each cohort is observed for at most five years of its life (1980s model years only at ages 32 to 45, 2010s at ages 2 to 16), and older cohorts contain only the vehicles that survived. Attrition is therefore compared within an era rather than across a full lifecycle; model years before 1981 are excluded upstream.
- Cohort counts weight every model equally, while 99.985% of peak registrations sit in cohorts with 100 or more vehicles, so cohort based results describe models rather than the vehicle population unless weighted by registrations.
- After these checks, `df_clean` has 262,052 rows and 14,007 cohorts.

## Feature Engineering

Create the features the models will use, starting with the attrition measures that define the dependent variable.

### Attrition Metrics

Calculate annual attrition for each cohort, along with vehicle age, peak registrations, cumulative attrition, and the change in the attrition rate.

In [ ]:
# annual periods: the same quarter in consecutive years, using April snapshots (April 2021 to April 2026 gives
# five periods that do not overlap); each row keeps the cohort's specifications and start of period shares
april = df_clean[df_clean['SNAPSHOT_DATE'].dt.month == 4]
end = (april[cohort_keys + ['SNAPSHOT_DATE', 'VIO_COUNT']]
       .assign(SNAPSHOT_DATE=lambda d: d['SNAPSHOT_DATE'] - pd.DateOffset(years=1))
       .rename(columns={'VIO_COUNT': 'VIO_END'}))
df_attrition = (april.merge(end, on=cohort_keys + ['SNAPSHOT_DATE'], how='inner')
                .rename(columns={'SNAPSHOT_DATE': 'PERIOD_START', 'VIO_COUNT': 'VIO_START'}))

# highest registrations seen up to the start of each period, from all quarterly snapshots (nothing later is used)
by_date = df_clean.sort_values(cohort_keys + ['SNAPSHOT_DATE'])
peak = by_date[cohort_keys + ['SNAPSHOT_DATE']].assign(PEAK_VIO_TO_DATE=by_date.groupby(cohort_keys)['VIO_COUNT'].cummax())
df_attrition = df_attrition.merge(peak.rename(columns={'SNAPSHOT_DATE': 'PERIOD_START'}), on=cohort_keys + ['PERIOD_START'])

df_attrition['AGE'] = df_attrition['PERIOD_START'].dt.year - df_attrition['YEAR_MODEL']
df_attrition['ANNUAL_ATTRITION'] = 1 - df_attrition['VIO_END'] / df_attrition['VIO_START']
df_attrition['CUMULATIVE_ATTRITION'] = 1 - df_attrition['VIO_START'] / df_attrition['PEAK_VIO_TO_DATE']

# keep periods that start at age 3 or older (registrations have stopped growing) with at least 100 vehicles,
# since smaller cohorts give unstable rates
eligible = (df_attrition['AGE'] >= 3) & (df_attrition['VIO_START'] >= 100)
print(f"periods: {len(df_attrition):,} built, {eligible.sum():,} kept "
      f"({(df_attrition['AGE'] < 3).sum():,} before age 3, {((df_attrition['AGE'] >= 3) & ~eligible).sum():,} under 100 vehicles)")
df_attrition = df_attrition[eligible].copy()

In [ ]:
# expected attrition: median for the same vehicle class, age, and period, using all eras together so cohorts near
# an era boundary share a baseline; excess attrition is how far a cohort is from that norm (negative = slower)
baseline_keys = ['VEHICLE_CLASS', 'AGE', 'PERIOD_START']
df_attrition['EXPECTED_ATTRITION'] = df_attrition.groupby(baseline_keys)['ANNUAL_ATTRITION'].transform('median')
df_attrition['EXCESS_ATTRITION'] = df_attrition['ANNUAL_ATTRITION'] - df_attrition['EXPECTED_ATTRITION']
print('fewest cohorts behind one expected value:', df_attrition.groupby(baseline_keys).size().min())

# previous period's values, matched by date so only consecutive kept periods are compared;
# a negative change means attrition slowed (the hypothesized collectability signal)
prior = (df_attrition[cohort_keys + ['PERIOD_START', 'ANNUAL_ATTRITION', 'EXCESS_ATTRITION']]
         .assign(PERIOD_START=lambda d: d['PERIOD_START'] + pd.DateOffset(years=1))
         .rename(columns={'ANNUAL_ATTRITION': 'PRIOR_ATTRITION', 'EXCESS_ATTRITION': 'PRIOR_EXCESS'}))
df_attrition = df_attrition.merge(prior, on=cohort_keys + ['PERIOD_START'], how='left')
df_attrition['ATTRITION_CHANGE'] = df_attrition['ANNUAL_ATTRITION'] - df_attrition['PRIOR_ATTRITION']
df_attrition['EXCESS_CHANGE'] = df_attrition['EXCESS_ATTRITION'] - df_attrition['PRIOR_EXCESS']

metric_cols = ['VIO_START', 'PEAK_VIO_TO_DATE', 'AGE', 'ANNUAL_ATTRITION', 'CUMULATIVE_ATTRITION', 'EXPECTED_ATTRITION',
               'EXCESS_ATTRITION', 'PRIOR_ATTRITION', 'ATTRITION_CHANGE', 'EXCESS_CHANGE']
print(f"df_attrition: {len(df_attrition):,} rows, {df_attrition.groupby(cohort_keys).ngroups:,} cohorts, "
      f"{df_attrition['PERIOD_START'].nunique()} periods")
df_attrition.groupby('VEHICLE_CLASS')[metric_cols].median().T.loc[metric_cols, class_order]

In [ ]:
# median annual attrition by age with the middle 80% of cohorts shaded
fig, ax = plt.subplots(figsize=(10, 4))
for cls in class_order:
    by_age = df_attrition[df_attrition['VEHICLE_CLASS'] == cls].groupby('AGE')['ANNUAL_ATTRITION']
    q = by_age.quantile([0.1, 0.5, 0.9]).unstack() * 100
    ax.plot(q.index, q[0.5], color=class_colors[cls], linewidth=2, label=f'{cls} (median)')
    ax.fill_between(q.index, q[0.1], q[0.9], color=class_colors[cls], alpha=0.15)
ax.axhline(0, color='#52514e', linewidth=0.8)
ax.set_xlabel('Vehicle Age at Period Start (years)')
ax.set_ylabel('Annual Attrition (%)')
ax.set_title('Annual Attrition by Vehicle Age (shaded: 10th to 90th percentile)')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# df_attrition.groupby(['VEHICLE_CLASS', 'AGE'])['ANNUAL_ATTRITION'].describe(percentiles=[0.1, 0.5, 0.9])

**Observations -**

- `df_attrition` has 53,102 cohort periods (31,914 car, 21,188 truck) from 11,427 cohorts across the five April to April periods; 9,824 cohorts have all five. Of the 63,819 periods built, 5,705 start before age 3 and 5,012 start below 100 vehicles, so they are excluded.
- Annual attrition rises with age to a peak at age 24 for cars (median 15.8%) and age 23 for trucks (10.8%), then falls to about 2% for cars and 3% for trucks by ages 40 to 44; trucks lose vehicles faster than cars from age 33 onward. 3.2% of periods show registrations growing (8.8% at age 35 and older), consistent with older vehicles being re-registered.
- Because every old cohort slows down, `EXPECTED_ATTRITION` (the median for the same class, age, and period, built from at least 53 cohorts) is the baseline, and `EXCESS_ATTRITION` measures distance from it; 80% of periods fall within about 4 points of their norm (10th percentile -4.2, 90th +4.5).
- `PRIOR_ATTRITION`, `PRIOR_EXCESS`, `ATTRITION_CHANGE`, and `EXCESS_CHANGE` need a previous kept period, so they exist for 41,666 rows. For predicting attrition (RQ2), only values known at the start of a period can be features (e.g. age, peak to date, prior attrition, prior excess); the current period's attrition, expected attrition, and changes are outcomes.
- `CUMULATIVE_ATTRITION` measures the decline from the highest registrations seen so far in the data, not from a cohort's original peak, which for older cohorts came before April 2021.

### Specification Features

Build features from vehicle specifications, such as grouped categories and values compared with other vehicles from the same model year.

In [ ]:
# one row per cohort (specifications do not change between snapshots); era sensitive values are compared
# with the same model year and vehicle class
specs = df_clean.drop_duplicates(cohort_keys).copy()
same_year = specs.groupby(['YEAR_MODEL', 'VEHICLE_CLASS'])
specs['MSRP_RANK'] = same_year['BASE_MSRP'].rank(pct=True)                                 # price position, free of inflation
specs['PRICE_SPREAD'] = specs['MAX_MSRP'] / specs['BASE_MSRP']                                # premium of the top version
specs['DISP_PER_1000_LBS'] = specs['MAX_ENGINE_DISP'] / specs['MAX_SHIP_WEIGHT'] * 1000      # power to weight stand in
specs['MANUAL_VS_ERA'] = specs['HAS_MANUAL'] - same_year['HAS_MANUAL'].transform('mean')     # manual rarity for its era
specs['RWD_VS_ERA'] = specs['HAS_RWD'] - same_year['HAS_RWD'].transform('mean')
specs['LOG_CONFIG_COUNT'] = np.log(specs['CONFIG_COUNT'])                                    # variety, right skewed

# grouped categories, so encoding creates a few dense columns instead of many sparse ones
body_family = {'Sedan': 'Sedan', 'Coupe': 'Coupe', 'Convertible': 'Convertible', 'Hatchback': 'Hatchback', 'Wagon': 'Wagon',
               'Sport Utility Vehicle': 'SUV', 'Sport Utility Truck': 'SUV', 'Pickup': 'Pickup', 'Incomplete Pickup': 'Pickup',
               'Van Passenger': 'Van', 'Van Cargo': 'Van', 'Van Camper': 'Van'}
specs['BODY_FAMILY'] = np.where(specs['BODY_STYLE_NAME'].isin(commercial_styles), 'Commercial',
                                specs['BODY_STYLE_NAME'].map(body_family).fillna('Other'))

def segment_family(name):
    if name in sport_segments:
        return 'Sport'
    for word in ['Commercial', 'Pickup', 'Van']:
        if word in name:
            return word
    if 'Utility' in name:
        return 'Luxury Utility' if 'Luxury' in name else 'Utility'
    if 'Luxury' in name or 'Prestige' in name:
        return 'Luxury'
    return 'Unknown' if name == 'Unknown' else 'Mainstream'
specs['SEGMENT_FAMILY'] = specs['SEGMENT_NAME'].map(segment_family)

no_engine_cohort = specs['PRIMARY_FUEL_TYPE'].isin(['E', 'R'])
specs['ENGINE_LAYOUT'] = np.where(no_engine_cohort, 'No Engine', specs['PRIMARY_ENGINE_BLOCK'].map(
    {'V': 'V', 'I': 'Inline', 'H': 'Flat', 'W': 'W', 'R': 'Rotary', 'Unknown': 'Unknown'}))
specs['FUEL_GROUP'] = specs['PRIMARY_FUEL_TYPE'].map({'G': 'Gas', 'D': 'Diesel', 'F': 'Flex', 'B': 'Hybrid',
                                                       'E': 'Electric'}).fillna('Other')

# truck only specifications; cars get their own category rather than a missing value
is_car = specs['VEHICLE_CLASS'] == 'Car'
specs['CAB_CONFIG'] = np.where(is_car, 'Not a Truck', specs['PRIMARY_CAB_CONFIG'].fillna('Unknown'))
specs['GVW_CLASS'] = np.where(is_car, 'Not a Truck', specs['PRIMARY_GVW_CLASS'].map(
    {1: 'Up to 6,000 lbs', 2: '6,001 to 10,000 lbs', 3: '10,001 to 14,000 lbs'}).fillna('Unknown'))

core_spec_cols = ['MSRP_RANK', 'PRICE_SPREAD', 'MAX_ENGINE_DISP', 'DISP_PER_1000_LBS', 'MAX_SHIP_WEIGHT', 'MIN_WHEELBASE_IN',
                  'MANUAL_VS_ERA', 'RWD_VS_ERA', 'HAS_FWD', 'HAS_AWD', 'HAS_4WD', 'HAS_REMOVABLE_ROOF', 'LOG_CONFIG_COUNT',
                  'BODY_FAMILY', 'SEGMENT_FAMILY', 'ENGINE_LAYOUT', 'FUEL_GROUP', 'ORIGIN_NAME', 'CAB_CONFIG', 'GVW_CLASS']
specs[core_spec_cols].describe(include='all').T[['count', 'unique', 'top', 'mean', 'min', 'max']]

In [ ]:
# optional candidates: brand home country (where the brand comes from, not where a vehicle is built), a performance
# trim flag, and specifications that only exist from model year 2000 onward
brand_country = {
    'Japan': ['ACURA', 'DAIHATSU', 'DATSUN', 'HINO', 'HONDA', 'INFINITI', 'ISUZU', 'LEXUS', 'MAZDA', 'MITSUBISHI',
              'MITSUBISHI FUSO TRUCK OF AMERICA INC', 'NISSAN', 'NISSAN DIESEL', 'SUBARU', 'SUZUKI', 'TOYOTA'],
    'Germany': ['AUDI', 'BMW', 'MAYBACH', 'MERCEDES-BENZ', 'MERKUR', 'PORSCHE', 'SMART', 'SPRINTER', 'VOLKSWAGEN'],
    'Italy': ['ALFA ROMEO', 'AUTOMOBILI PININFARINA', 'FERRARI', 'FIAT', 'IVECO', 'LAFORZA', 'LAMBORGHINI', 'LANCIA', 'MASERATI'],
    'United Kingdom': ['ASTON MARTIN', 'BENTLEY', 'INEOS AUTOMOTIVE LIMITED', 'JAGUAR', 'LAND ROVER', 'LOTUS',
                       'MCLAREN AUTOMOTIVE', 'MINI', 'ROLLS-ROYCE', 'STERLING', 'TRIUMPH CAR', 'TVR'],
    'South Korea': ['DAEWOO', 'GENESIS', 'HYUNDAI', 'KIA'],
    'Sweden': ['POLESTAR', 'SAAB', 'VOLVO'],
    'France': ['BUGATTI', 'PEUGEOT', 'RENAULT'],
    'Other': ['AZURE DYNAMICS', 'BUGATTI RIMAC', 'BYD COACH AND BUS LLC', 'KARSAN OTOMOTIV SANAYII VE TIC. A.S.',
              'VINFAST TRADING AND PRODUCTION LLC', 'YUGO', 'ZEEKR'],
}
make_country = {make: country for country, makes in brand_country.items() for make in makes}
specs['BRAND_COUNTRY'] = specs['MAKE'].map(make_country).fillna('United States')   # every other make is an American brand

performance_badges = ['SS', 'SRT', 'HELLCAT', 'SCAT PACK', 'AMG', 'NISMO', 'STI', 'WRX', 'TRD PRO', 'SHELBY', 'COBRA', 'SVT',
                      'SVO', 'BOSS', 'MACH 1', 'RAPTOR', 'LIGHTNING', 'TRX', 'Z28', 'IROC', 'Z06', 'ZL1', 'ZR1', 'TYPE R',
                      'TYPE S', 'GTI', 'SI', 'GSR', 'VR-4', 'T-TYPE', 'GNX', 'GRAND NATIONAL', 'TRANS AM', 'FORMULA', 'GT2',
                      'GT3', 'GTS', 'QUADRIFOGLIO', 'COMPETITION', 'TURBO S']
badge_pattern = r'(?<![A-Z0-9])(?:' + '|'.join(re.escape(b) for b in performance_badges) + r')(?![A-Z0-9])'
specs['PERFORMANCE_TRIM'] = specs['TRIM_NAMES'].str.contains(badge_pattern, regex=True).astype(float).where(specs['TRIM_NAMES'].notna())
specs['HP_RANK'] = same_year['MAX_HORSEPOWER'].rank(pct=True)
specs['LBS_PER_HP'] = specs['MAX_SHIP_WEIGHT'] / specs['MAX_HORSEPOWER']
specs['ENGINE_LOCATION'] = specs['PRIMARY_ENGINE_LOCATION'].map({'FR': 'Front', 'MI': 'Mid', 'RR': 'Rear'}).fillna('Unknown')
specs['TWO_SEATER'] = (specs['MIN_SEATS'] <= 2).astype(float).where(specs['MIN_SEATS'].notna())
specs['FORCED_INDUCTION'] = specs[['HAS_TURBO', 'HAS_SUPERCHARGER']].max(axis=1)

# coverage before any filling (from df_cohort), for the cohorts in df_attrition, by the era each candidate would serve
era_order = ['1981 to 1999', '2000 to 2009', '2010 onward']
raw = df_cohort.drop_duplicates(cohort_keys).set_index(cohort_keys).loc[
    df_attrition.drop_duplicates(cohort_keys).set_index(cohort_keys).index]
raw_era = pd.cut(raw.index.get_level_values('YEAR_MODEL'), [1980, 1999, 2009, 2100], labels=era_order)
everyone = pd.Series(True, index=raw.index)
combustion, trucks = ~raw['PRIMARY_FUEL_TYPE'].isin(['E', 'R']), raw['PRIMARY_VEH_TYPE'] == 'T'
candidates = {   # candidate: (has a real value, cohorts it applies to)
    'BRAND_COUNTRY': (everyone, everyone),
    'PERFORMANCE_TRIM': (raw['TRIM_NAMES'].notna(), everyone),
    'HP_RANK': (raw['MAX_HORSEPOWER'].notna(), everyone),
    'LBS_PER_HP': (raw['MAX_HORSEPOWER'].notna() & raw['MAX_SHIP_WEIGHT'].notna(), everyone),
    'ENGINE_LOCATION': (raw['PRIMARY_ENGINE_LOCATION'].notna(), everyone),
    'TWO_SEATER': (raw['MIN_SEATS'].notna(), everyone),
    'MIN_HEIGHT_IN': (raw['MIN_HEIGHT_IN'].notna(), everyone),
    'FORCED_INDUCTION': (raw[['HAS_TURBO', 'HAS_SUPERCHARGER']].notna().any(axis=1), everyone),
    'MAX_MPG_COMBINED': (raw['MAX_MPG_COMBINED'].notna(), combustion),
    'MAX_TOW_CAPACITY': (raw['MAX_TOW_CAPACITY'].notna(), trucks),
    'MAX_PAYLOAD': (raw['MAX_PAYLOAD'].notna(), trucks),
}
coverage = pd.DataFrame({name: (has & applies).groupby(raw_era, observed=True).sum() / applies.groupby(raw_era, observed=True).sum() * 100
                         for name, (has, applies) in candidates.items()}).T.round(1)

# keep a candidate when at least 80% of the cohorts it applies to have a real value in every era that uses it;
# 2000 onward specifications are judged on the two later eras only, and the trim flag is also excluded because
# matching badge words in trim names misclassifies some versions
eras_used = {name: era_order if name in ['BRAND_COUNTRY', 'PERFORMANCE_TRIM'] else era_order[1:] for name in candidates}
coverage['MIN_IN_ERAS_USED'] = [coverage.loc[name, eras_used[name]].min() for name in coverage.index]
coverage['KEEP'] = (coverage['MIN_IN_ERAS_USED'] >= 80) & (coverage.index != 'PERFORMANCE_TRIM')
coverage

**Observations -**

- 20 core specification features are built for all 14,007 cohorts: era relative price and drivetrain (`MSRP_RANK`, `MANUAL_VS_ERA`, `RWD_VS_ERA`), the top version premium (`PRICE_SPREAD`, median 1.21, 2.31 at the 99th percentile), displacement per 1,000 lbs, size, drive flags, and grouped categories (10 body families, 9 segment families, 7 engine layouts, 6 fuel groups, truck cab and weight class). Cylinders, door count, and the two vehicle type columns are left out under the selection rule (more data first, then the better known attribute).
- Only `MANUAL_VS_ERA` (15.7% of cohorts unknown), `HAS_REMOVABLE_ROOF` (24.0%), and the drive flags (4 cohorts) still have missing values; they stay missing so unknown is not read as "no", which the gradient boosted trees handle natively.
- Under the 80% rule, the kept optional features are brand country (100%), horsepower rank (86.1% and 91.7% in the two later eras), pounds per horsepower (85.9% and 90.2%), and engine location (86.1% and 87.3%). Excluded: the performance trim flag (71.8% before 2000, and badge matching misclassifies some versions), two seater (77.2% in 2000 to 2009), height (77.1%), turbo or supercharger (32.7%), MPG (66.4%), towing (70.6%), and payload (54.8%).
- Brand country follows the brand's heritage, not where a vehicle is built (a Honda built in Ohio is Japan): United States 5,467 cohorts, Japan 3,554, Germany 2,525, United Kingdom 878, South Korea 630, Italy 508, Sweden 372, France 46, and Other 27. Horsepower rank and pounds per horsepower use the values filled in Missing Values for the 8% to 14% of later era cohorts without horsepower.

## Era Split

Divide cohorts by model year era. This happens after features are calculated on the full dataset, because the available specification columns differ by era.

In [ ]:
# eras follow specification availability: 2000 onward columns start in 2000 and turbo data fills in from 2010;
# 2020s cohorts (ages 3 to 5 only) join 2010 onward
keep_optional = coverage.index[coverage['KEEP']].tolist()
all_era_optional = [c for c in keep_optional if eras_used[c] == era_order]
later_era_optional = [c for c in keep_optional if c not in all_era_optional]
model_specs = core_spec_cols + all_era_optional + later_era_optional
df_features = df_attrition.merge(specs[cohort_keys + [c for c in model_specs if c not in df_attrition.columns]],
                                 on=cohort_keys, how='left', validate='many_to_one')
df_features['ERA'] = pd.cut(df_features['YEAR_MODEL'], [1980, 1999, 2009, 2100], labels=era_order)

# inputs known at the start of a period: behavior so far, start of period VIO shares, and specifications
behavior_cols = ['AGE', 'VIO_START', 'CUMULATIVE_ATTRITION', 'PRIOR_ATTRITION', 'PRIOR_EXCESS',
                 'SHARE_CONVERTIBLE', 'SHARE_COUPE', 'SHARE_REGULAR_CAB', 'SHARE_WORK_BODY', 'SHARE_PERSONAL']
df_features['VIO_START'] = np.log(df_features['VIO_START'])   # right skewed; log scale
era_features = {era: ['VEHICLE_CLASS'] + behavior_cols + core_spec_cols + all_era_optional
                     + (later_era_optional if era != era_order[0] else []) for era in era_order}
print('optional features kept:', keep_optional)
df_features.groupby('ERA', observed=True).agg(PERIODS=('AGE', 'size'), COHORTS=('MODEL', lambda m: df_features.loc[m.index, cohort_keys].drop_duplicates().shape[0]),
                                              MODEL_YEARS=('YEAR_MODEL', lambda y: f'{y.min()} to {y.max()}'),
                                              AGES=('AGE', lambda a: f'{a.min()} to {a.max()}'))

**Observations -**

- 1981 to 1999: 20,423 periods from 4,136 cohorts (ages 22 to 44); 2000 to 2009: 14,421 periods from 2,896 cohorts (ages 12 to 25); 2010 onward: 18,258 periods from 4,395 cohorts (model years 2010 to 2022, ages 3 to 15).
- Every era uses vehicle class, 10 inputs known at the start of a period (age, log registrations, cumulative attrition, prior attrition and excess, and the five VIO shares), the 20 core specifications, and brand country (32 features); the two later eras add horsepower rank, pounds per horsepower, and engine location (35 features).
- Attrition metrics and the expected attrition baseline were calculated on all eras together before this split, so cohorts near an era boundary share the same norm.

## One-Hot Encoding

Convert the selected categorical columns into 0/1 columns so models can use them. Encoding is done separately for each era, since category lists differ.

In [ ]:
# one hot encoding, separately for each era so each era has only its own categories; every category keeps its own
# column (tree models do not need one dropped, and the encoded columns are grouped back for significance)
categorical_cols = ['VEHICLE_CLASS', 'BODY_FAMILY', 'SEGMENT_FAMILY', 'ENGINE_LAYOUT', 'FUEL_GROUP', 'ORIGIN_NAME',
                    'CAB_CONFIG', 'GVW_CLASS', 'BRAND_COUNTRY', 'ENGINE_LOCATION']
encoded = {}
for era in era_order:
    rows = df_features[df_features['ERA'] == era]
    cats = [c for c in categorical_cols if c in era_features[era]]
    encoded[era] = pd.get_dummies(rows[era_features[era]], columns=cats, dtype=int)
    print(f"{era}: {len(era_features[era])} features ({len(cats)} categorical) -> {encoded[era].shape[1]} columns")

# example: body family becomes one 0/1 column per category
encoded[era_order[0]].filter(like='BODY_FAMILY_').head()

**Observations -**

- The 10 categorical variables (9 in 1981 to 1999, which has no engine location) expand to 81, 88, and 89 columns, and each era only gets columns for the categories it contains.
- Every category keeps its own column because tree models do not need a reference category dropped (a linear model would drop one per variable). Unknown values are their own column (e.g. `SEGMENT_FAMILY_Unknown`), and cars get a `Not a Truck` column for truck specifications instead of a missing value.

## Variable Significance Analysis

Measure which features matter most for predicting annual attrition and excess attrition in each era.

In [ ]:
# gradient boosted trees per era: train on the first four periods, test on the held out final period
# (April 2025 to April 2026), so each model is judged on a year it has not seen. Three runs per era:
# annual attrition (RQ2), excess attrition, and excess attrition without the cohort's attrition history, so that
# history (which already reflects a cohort's specifications) does not mask the specification effects
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

history_cols = ['PRIOR_ATTRITION', 'PRIOR_EXCESS', 'CUMULATIVE_ATTRITION']
runs = [('ANNUAL_ATTRITION', 'ANNUAL_ATTRITION', False), ('EXCESS_ATTRITION', 'EXCESS_ATTRITION', False),
        ('EXCESS_ATTRITION (specs only)', 'EXCESS_ATTRITION', True)]
test_start = df_features['PERIOD_START'].max()

def grouped_permutation_importance(model, X, y, groups, n_repeats=5, seed=0):
    # drop in R^2 when one original feature (all of its encoded columns together) is shuffled
    rng = np.random.default_rng(seed)
    base = r2_score(y, model.predict(X))
    drops = {}
    for name, cols in groups.items():
        scores = []
        for _ in range(n_repeats):
            shuffled = X.copy()
            shuffled[cols] = X[cols].to_numpy()[rng.permutation(len(X))]
            scores.append(base - r2_score(y, model.predict(shuffled)))
        drops[name] = np.mean(scores)
    return pd.Series(drops)

results, importance = [], {}
for era in era_order:
    rows = df_features.loc[encoded[era].index]
    train, test = rows['PERIOD_START'] < test_start, rows['PERIOD_START'] == test_start
    for label, target, specs_only in runs:
        features = [f for f in era_features[era] if not (specs_only and f in history_cols)]
        X = encoded[era].drop(columns=history_cols if specs_only else [])
        groups = {f: [c for c in X.columns if c == f or c.startswith(f + '_')] for f in features}
        y = rows[target]
        model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0).fit(X[train], y[train])
        pred = model.predict(X[test])
        # simple benchmark: the training median for the same class and age (zero for excess attrition)
        bench = (rows[train].groupby(['VEHICLE_CLASS', 'AGE'])[target].median()
                 .reindex(pd.MultiIndex.from_frame(rows.loc[test, ['VEHICLE_CLASS', 'AGE']])).fillna(y[train].median()).to_numpy()
                 if target == 'ANNUAL_ATTRITION' else np.zeros(test.sum()))
        results.append({'ERA': era, 'RUN': label, 'TRAIN_ROWS': int(train.sum()), 'TEST_ROWS': int(test.sum()),
                        'TEST_R2': r2_score(y[test], pred), 'TEST_MAE_PTS': mean_absolute_error(y[test], pred) * 100,
                        'BENCHMARK_MAE_PTS': mean_absolute_error(y[test], bench) * 100})
        importance[(era, label)] = grouped_permutation_importance(model, X[test], y[test], groups)

pd.DataFrame(results).round(3)

In [ ]:
# ten most important features per era and run (drop in test R^2 when the feature is shuffled)
run_colors = ['#2a78d6', '#eb6834', '#52514e']
fig, axes = plt.subplots(len(runs), len(era_order), figsize=(16, 12))
for i, (label, _, _) in enumerate(runs):
    for j, era in enumerate(era_order):
        top = importance[(era, label)].sort_values(ascending=False).head(10)[::-1]
        ax = axes[i, j]
        ax.barh(top.index, top.values, color=run_colors[i])
        ax.set_title(f'{era}: {label}', fontsize=10)
        ax.tick_params(axis='y', labelsize=8)
        ax.spines[['top', 'right']].set_visible(False)
fig.supxlabel('Drop in Test R² When Shuffled')
fig.suptitle('Variable Significance by Era and Run (gradient boosted trees, permutation importance)')
plt.tight_layout()
plt.show()

# full ranking: pd.DataFrame(importance).round(4)

In [ ]:
# direction of the leading specification effects: median excess attrition in points by era
# (negative = losing vehicles more slowly than cohorts of the same class, age, and period)
f = df_features
personal_rank = f.groupby('ERA', observed=True)['SHARE_PERSONAL'].rank(pct=True)   # within era
checks = {
    'Body: convertible': f['BODY_FAMILY'] == 'Convertible', 'Body: coupe': f['BODY_FAMILY'] == 'Coupe',
    'Body: sedan': f['BODY_FAMILY'] == 'Sedan', 'Segment: sport': f['SEGMENT_FAMILY'] == 'Sport',
    'Segment: mainstream': f['SEGMENT_FAMILY'] == 'Mainstream',
    'Cars: RWD offered': (f['VEHICLE_CLASS'] == 'Car') & (f['HAS_RWD'] == 1),
    'Cars: RWD not offered': (f['VEHICLE_CLASS'] == 'Car') & (f['HAS_RWD'] == 0),
    'MSRP rank in top 20% of model year': f['MSRP_RANK'] >= 0.8, 'MSRP rank in bottom 20%': f['MSRP_RANK'] < 0.2,
    'Personal use share in top 20% of era': personal_rank >= 0.8, 'Personal use share in bottom 20%': personal_rank < 0.2,
    'Brand: Italy': f['BRAND_COUNTRY'] == 'Italy', 'Brand: United Kingdom': f['BRAND_COUNTRY'] == 'United Kingdom',
    'Brand: Japan': f['BRAND_COUNTRY'] == 'Japan', 'Brand: United States': f['BRAND_COUNTRY'] == 'United States',
    'Engine: mid (2000 onward)': f['ENGINE_LOCATION'] == 'Mid',
}
pd.DataFrame({name: f[mask].groupby('ERA', observed=True)['EXCESS_ATTRITION'].median() * 100
              for name, mask in checks.items()}).T.round(2)

**Observations -**

- All nine models beat the benchmark (the training median for the same class and age) on the held out final period. With history, test R² is 0.79, 0.91, and 0.53 for annual attrition and 0.67, 0.92, and 0.42 for excess attrition (1981 to 1999, 2000 to 2009, 2010 onward), with errors of 0.7 to 1.6 points against 1.4 to 4.5 for the benchmark.
- With history included, the previous period's attrition or excess dominates every ranking (in 2000 to 2009, shuffling it drops R² by more than 1), so attrition is highly persistent and specifications add little once history is known. Age is the other major driver of annual attrition in 1981 to 1999 and 2010 onward.
- Without history, specifications explain excess attrition about as well (R² 0.68, 0.89, and 0.48), so slower or faster than normal attrition is largely structural. The leading drivers are personal use share, max displacement, RWD relative to era, and segment (1981 to 1999); personal use share, FWD, segment, displacement, and MSRP rank (2000 to 2009); and MSRP rank, brand country, personal use share, and displacement (2010 onward), with age present in each.
- The direction supports the hypothesis: cohorts with enthusiast specifications lose vehicles more slowly than their peers in every era, most strongly in 2000 to 2009. Convertibles sit -4.6, -6.2, and -1.0 points from their norm, sport segments -1.9, -5.6, and -1.0, cars offering RWD -0.4, -3.0, and -0.2 (against +0.6, +2.4, and +0.1 without), Italian brands -2.9, -6.6, and -0.2, and mid engine cars -8.0 and -1.0, while sedans and mainstream segments run above their norm.
- Personal use share ranks highly but is not one directional (cohorts in both the top and bottom 20% of their era sit below the norm). Differences shrink in 2010 onward, where cohorts are young and excess attrition varies least (standard deviation 1.9 points against 4.0 and 5.2). These are associations within the observed periods, not causes, and they feed the clustering (RQ1) and modeling in the next milestone.

## Summary of Features and Transformations

This cell summarizes the features created and the transformations applied.

### Features Created

- **Dependent variables (Attrition Metrics):** `ANNUAL_ATTRITION`, the share of a cohort's registrations lost from one April to the next (the RQ2 target), and `EXCESS_ATTRITION`, annual attrition minus `EXPECTED_ATTRITION` (the median for the same class, age, and period). Excess attrition is the collectability signal: every older cohort slows down, so only slower than normal attrition is distinctive.
- **History known at the start of a period:** `AGE`, log `VIO_START`, `PEAK_VIO_TO_DATE`, `CUMULATIVE_ATTRITION` (decline from the highest registrations so far), and `PRIOR_ATTRITION` and `PRIOR_EXCESS` (persistence). `ATTRITION_CHANGE` and `EXCESS_CHANGE` measure the slowdown itself and are kept as outcomes, not model inputs.
- **VIO registration shares at the start of a period:** convertible, coupe, regular cab, work body, and personal use, which count registered vehicles rather than VIN patterns.
- **Era relative specifications:** `MSRP_RANK` (price within model year and class, removing inflation), `MANUAL_VS_ERA` and `RWD_VS_ERA` (availability compared with the model year's norm), and `HP_RANK` (2000 onward).
- **Performance, size, and variety:** `PRICE_SPREAD` (premium of the top version), `DISP_PER_1000_LBS` (power to weight for all eras), `LBS_PER_HP` and `ENGINE_LOCATION` (2000 onward), displacement, weight, wheelbase, drive flags, removable roof, and `LOG_CONFIG_COUNT`.
- **Grouped categories, one-hot encoded per era (81, 88, and 89 columns):** body family, segment family, engine layout, fuel group, origin, truck cab configuration and weight class, brand home country, and vehicle class.
- **Left out:** cylinders, door count, and the duplicate vehicle type columns (where attributes overlap, the one with more data, or the better known one, was kept), and optional candidates with under 80% real data in an era that would use them (performance trim flag, two seater, height, turbo or supercharger, MPG, towing, payload).

### Null Value Management

- The 513 electric and hydrogen cohorts without displacement were given zero displacement and cylinders rather than an imputed engine; their MPG stays missing because MPG does not apply.
- Other numeric specifications were filled with the median of the same model year and vehicle class (e.g. price 6.0% and shipping weight 8.1% missing), so filled values match their era. Groups with no data at all (horsepower before 2000, towing for most cars) stay missing and are handled by the era split.
- Missing categories became an explicit "Unknown" category, and truck specifications are "Not a Truck" for cars. Yes/no flags without data (e.g. manual offered, 17.6% of cohorts) stay missing so unknown is never read as "no"; the gradient boosted trees handle missing values natively.
- Prior period values are missing for each cohort's first kept period (20% to 24% of rows per era) and are left missing.

### Outlier Treatment

- Specification errors were found by comparing each value with the same model's neighboring model years (which must agree within 20%) and replaced with the neighbors' median: 35 values in 34 cohorts, including 27 shipping weights (e.g. the 2011 Lexus LS at 7,420 lbs), impossible heights and wheelbases, a 2,024 hp 2024 Lexus UX, and 38 hp for the 2020 BMW i3. Errors repeated in consecutive years (e.g. the 2010 and 2011 Lotus Elise wheelbase) are not caught.
- Real extremes such as hypercar prices and horsepower were kept; skewed values were transformed (ranks within model year, logs of registrations and configuration count) rather than capped.
- For attrition, periods starting below 100 vehicles (5,012) or before age 3 (5,705) were excluded, because small cohorts give unstable rates and young cohorts are still growing; registration growth in older cohorts (3.2% of periods) was kept as real negative attrition.

### Sampling Bias

- Models sold only outside the US (330 cohorts with a median of 4 registered vehicles) and the 2007 Subaru Legacy and Outback (records moved between them in April 2024) were removed.
- Cohorts below 100 vehicles are 7.6% of cohorts but 0.013% of registrations, led by exotic and luxury makes (Ferrari, Rolls-Royce, Bentley), so the attrition models underrepresent vehicles that are collectible by design.
- Each cohort is observed for at most five years of its life, and older cohorts contain only the vehicles that survived; the era split compares cohorts at similar life stages, and expected attrition compares each cohort with peers of the same class, age, and period. Model years before 1981 (no specifications) are out of scope.
- Results count models equally, not vehicles; weighting by registrations would shift them toward high volume models.

### Non-Standard Value Formatting

- Upstream in Snowflake: numbers stored as text were converted, the RWD code was corrected, blanks and zero placeholders became missing, plant country spellings were standardized, and "most common" values use a fixed, neutral tie break so rebuilds are repeatable.
- In this notebook: coded categories got readable labels; MPGe values for electric and hydrogen vehicles (372 cohorts) and the GMC Hummer EV's wheel torque (8 cohorts) were removed because they use different units from the rest of their columns; and vehicle class comes from VINtelligence, which is fixed per cohort, rather than VIO's vehicle type, which disagrees for 85 rows.

## Model Development and Evaluation

Milestone 4 compares at least two modeling techniques for each analytical task, refines each one, and selects the model that best serves the business problem:

- **Attrition regression (RQ2):** predict a cohort's annual attrition with gradient boosted trees and ridge regression.
- **Collectability classification:** predict whether a cohort will be an outperformer (excess attrition in the lowest 20% of its era for the period) with gradient boosted trees and logistic regression.
- **Generalization check:** retest the chosen models on model families they never saw in training.
- **Vehicle clustering (RQ1):** group cohorts by specification profile with k-means and hierarchical clustering, then compare the groups' attrition patterns.

Models are fit separately for each era, matching Milestone 3. Every setting (inputs, parameters, decision threshold) is chosen on the 2024 test year (trained on 2021 to 2023), then confirmed on 2025 (trained on 2021 to 2024), a year no setting was tuned on.

In [ ]:
# shared setup: models, feature selection, and time based splits (train on earlier periods, test on a later one)
import ast
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (calinski_harabasz_score, confusion_matrix, davies_bouldin_score, f1_score, mean_squared_error,
                             precision_score, recall_score, roc_auc_score, silhouette_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def linear_pipeline(estimator):
    # linear models need complete inputs on one scale: gaps are filled with the median (plus a 0/1 column marking
    # each gap, so the model still knows a value was missing) and every column is standardized
    return make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler(), estimator)

def columns_for(era, features):
    return [c for c in encoded[era].columns if any(c == f or c.startswith(f + '_') for f in features)]

def split(era, features, test_year):
    # train on every period starting before April of the test year, test on the period starting in that April
    X = encoded[era][columns_for(era, features)]
    rows = df_features.loc[X.index]
    year = rows['PERIOD_START'].dt.year
    return X, rows, year < test_year, year == test_year

for era in era_order:
    # attrition rises and falls with age; a squared age term lets linear models follow that curve (trees do not need it)
    encoded[era]['SQUARED_AGE'] = encoded[era]['AGE'] ** 2

def input_sets(era, top_features):
    return {'All inputs': era_features[era],
            'All inputs + squared age': era_features[era] + ['SQUARED_AGE'],
            'Specs only': [f for f in era_features[era] if f not in history_cols],
            'Top 10 from Milestone 3': top_features}

test_years = [2024, 2025]   # 2024 is used to choose settings; 2025 is the final, untouched test

### Attrition Regression (RQ2)

Gradient boosted trees against ridge regression (a linear model with a penalty that keeps coefficients stable when inputs overlap), refined by test year, input set, and parameters, and judged by mean absolute error (MAE) and R².

In [ ]:
# regression (RQ2): predict annual attrition with gradient boosted trees and ridge regression, refining each by
# test year (2024, 2025), inputs (all, specs only, Milestone 3 top 10), and model parameters
reg_target = 'ANNUAL_ATTRITION'
reg_models = {'Gradient boosted trees': lambda **p: HistGradientBoostingRegressor(max_iter=300, random_state=0, **p),
              'Ridge regression': lambda **p: linear_pipeline(Ridge(**p))}
reg_params = {'Gradient boosted trees': [{'learning_rate': lr, 'max_leaf_nodes': n} for lr in (0.03, 0.1) for n in (15, 63)],
              'Ridge regression': [{'alpha': a} for a in (1, 10, 100, 1000)]}

reg_rows = []
for era in era_order:
    inputs = input_sets(era, importance[(era, 'ANNUAL_ATTRITION')].nlargest(10).index.tolist())
    for test_year in test_years:
        # benchmark: the training median for the same class and age
        X, rows, train, test = split(era, inputs['All inputs'], test_year)
        bench = (rows[train].groupby(['VEHICLE_CLASS', 'AGE'])[reg_target].median()
                 .reindex(pd.MultiIndex.from_frame(rows.loc[test, ['VEHICLE_CLASS', 'AGE']])).fillna(rows.loc[train, reg_target].median()))
        reg_rows.append({'ERA': era, 'MODEL': 'Benchmark', 'INPUTS': 'Class and age', 'PARAMS': '', 'TEST_YEAR': test_year,
                         'MAE_PTS': mean_absolute_error(rows.loc[test, reg_target], bench) * 100,
                         'R2': r2_score(rows.loc[test, reg_target], bench)})
        for name, make in reg_models.items():
            for input_name, features in inputs.items():
                X, rows, train, test = split(era, features, test_year)
                for params in reg_params[name]:
                    pred = make(**params).fit(X[train], rows.loc[train, reg_target]).predict(X[test])
                    reg_rows.append({'ERA': era, 'MODEL': name, 'INPUTS': input_name, 'PARAMS': str(params), 'TEST_YEAR': test_year,
                                     'MAE_PTS': mean_absolute_error(rows.loc[test, reg_target], pred) * 100,
                                     'R2': r2_score(rows.loc[test, reg_target], pred)})
reg_results = pd.DataFrame(reg_rows)

# for each model and input set, keep the parameters with the lowest 2024 error, then report both test years
val = reg_results[reg_results['TEST_YEAR'] == 2024]
chosen = val.loc[val.groupby(['ERA', 'MODEL', 'INPUTS'])['MAE_PTS'].idxmin(), ['ERA', 'MODEL', 'INPUTS', 'PARAMS']]
reg_summary = (chosen.merge(reg_results, on=['ERA', 'MODEL', 'INPUTS', 'PARAMS'])
               .pivot_table(index=['ERA', 'MODEL', 'INPUTS', 'PARAMS'], columns='TEST_YEAR', values=['MAE_PTS', 'R2'])
               .round(3))
reg_summary

In [ ]:
# parameter sensitivity on the 2024 test year (all inputs): how much each setting changes the error, in points
(reg_results[(reg_results['TEST_YEAR'] == 2024) & (reg_results['INPUTS'] == 'All inputs') & (reg_results['MODEL'] != 'Benchmark')]
 .pivot_table(index=['MODEL', 'PARAMS'], columns='ERA', values='MAE_PTS', observed=True).round(3))

In [ ]:
# final comparison on 2025: each model's best input set and parameters (chosen on 2024) against the benchmark
best_reg = (reg_summary.reset_index().sort_values(('MAE_PTS', 2024))
            .groupby(['ERA', 'MODEL'], sort=False).head(1).set_index(['ERA', 'MODEL']))
best_reg = best_reg.loc[[(e, m) for e in era_order for m in ['Gradient boosted trees', 'Ridge regression', 'Benchmark']]]
reg_choice = {(e, m): (r[('INPUTS', '')], r[('PARAMS', '')]) for (e, m), r in best_reg.iterrows() if m != 'Benchmark'}

fig, ax = plt.subplots(figsize=(10, 4))
model_colors = {'Gradient boosted trees': '#2a78d6', 'Ridge regression': '#eb6834', 'Benchmark': '#b8b6b0'}
width = 0.27
for k, model in enumerate(model_colors):
    vals = [best_reg.loc[(e, model), ('MAE_PTS', 2025)] for e in era_order]
    bars = ax.bar(np.arange(len(era_order)) + (k - 1) * width, vals, width, color=model_colors[model], label=model)
    ax.bar_label(bars, fmt='%.2f', fontsize=8, padding=2)
ax.set_xticks(np.arange(len(era_order)), era_order)
ax.set_ylabel('Mean Absolute Error (points)')
ax.set_title('Annual Attrition Prediction Error on the 2025 Test Year (lower is better)')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

best_reg[[('INPUTS', ''), ('PARAMS', ''), ('MAE_PTS', 2024), ('MAE_PTS', 2025), ('R2', 2024), ('R2', 2025)]]

**Observations -**

- **Inputs mattered more than parameters.** For gradient boosted trees, all inputs worked best in 1981 to 1999 and 2000 to 2009, and specifications only in 2010 onward (history adds little for young cohorts). Ridge regression gained from squared age, because attrition curves with age (2025 MAE 2.01 to 1.90 points in 1981 to 1999, 1.79 to 1.60 in 2000 to 2009), and lost the most from dropping history (2.36, 2.91, and 1.18 points with specifications only).
- **With all inputs, parameters changed error by less than 0.1 points for either model.** The best ridge penalty sits at the top of the grid (1,000) in two eras, but the error barely moves across penalties.
- **Results hold across two future years.** Errors on 2024 and 2025 are close for both models (trees: 1.57 and 1.64, 1.03 and 1.09, 0.75 and 0.84 points), which addresses RQ2's question about predicting across multiple annual periods.
- **Verdict: gradient boosted trees performed better.** On 2025, trees have the lowest error in every era (1.64, 1.09, and 0.84 points against 1.90, 1.47, and 1.09 for ridge and 2.88, 4.48, and 1.50 for the benchmark) and higher or equal R² (0.79 vs 0.73, 0.91 vs 0.86, 0.59 vs 0.58). Attrition depends on age and specifications in curved, interacting ways that trees capture directly and a linear model can only approximate; ridge's advantage is readable coefficients.

### Collectability Classification

Gradient boosted trees against logistic regression for predicting outperformers, refined by test year, input set, parameters, and decision threshold, and judged by ROC AUC, precision, recall, F1, the confusion matrix (Type I and Type II errors), and watch list precision (the share of true outperformers among the 10% of cohorts scored highest).

In [ ]:
# collectability classification: a period is a 'yes' when the cohort's excess attrition is in the lowest 20% of its
# era for that period, meaning it lost vehicles clearly more slowly than its peers
low_cut = df_features.groupby(['ERA', 'PERIOD_START'], observed=True)['EXCESS_ATTRITION'].transform(lambda s: s.quantile(0.2))
df_features['OUTPERFORMER'] = (df_features['EXCESS_ATTRITION'] <= low_cut).astype(int)
cls_target = 'OUTPERFORMER'

cls_models = {'Gradient boosted trees': lambda **p: HistGradientBoostingClassifier(max_iter=300, random_state=0, **p),
              'Logistic regression': lambda **p: linear_pipeline(LogisticRegression(max_iter=2000, **p))}
cls_params = {'Gradient boosted trees': [{'learning_rate': lr, 'max_leaf_nodes': n} for lr in (0.03, 0.1) for n in (15, 63)],
              'Logistic regression': [{'C': c} for c in (0.01, 0.1, 1, 10)]}

cls_rows, cls_scores = [], {}
for era in era_order:
    inputs = input_sets(era, importance[(era, 'EXCESS_ATTRITION')].nlargest(10).index.tolist())
    for name, make in cls_models.items():
        for input_name, features in inputs.items():
            for test_year in test_years:
                X, rows, train, test = split(era, features, test_year)
                for params in cls_params[name]:
                    prob = make(**params).fit(X[train], rows.loc[train, cls_target]).predict_proba(X[test])[:, 1]
                    cls_scores[(era, name, input_name, str(params), test_year)] = prob
                    cls_rows.append({'ERA': era, 'MODEL': name, 'INPUTS': input_name, 'PARAMS': str(params), 'TEST_YEAR': test_year,
                                     'AUC': roc_auc_score(rows.loc[test, cls_target], prob)})
cls_results = pd.DataFrame(cls_rows)

# keep the parameters with the highest 2024 AUC for each model and input set, then report both test years
val = cls_results[cls_results['TEST_YEAR'] == 2024]
cls_chosen = val.loc[val.groupby(['ERA', 'MODEL', 'INPUTS'])['AUC'].idxmax(), ['ERA', 'MODEL', 'INPUTS', 'PARAMS']]
(cls_chosen.merge(cls_results, on=['ERA', 'MODEL', 'INPUTS', 'PARAMS'])
 .pivot_table(index=['ERA', 'MODEL', 'INPUTS', 'PARAMS'], columns='TEST_YEAR', values='AUC').round(3))

In [ ]:
# final classifiers: the best input set and parameters on 2024 AUC; the decision threshold is also set on 2024
# (the one with the highest F1), then applied unchanged to 2025
def class_metrics(y, prob, threshold):
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    top = prob >= np.quantile(prob, 0.9)   # the 10% of cohorts scored highest: a watch list
    return {'THRESHOLD': threshold, 'AUC': roc_auc_score(y, prob), 'PRECISION': precision_score(y, pred),
            'RECALL': recall_score(y, pred), 'F1': f1_score(y, pred), 'FALSE_POS_TYPE_I': fp, 'FALSE_NEG_TYPE_II': fn,
            'TRUE_POS': tp, 'TRUE_NEG': tn, 'WATCH_LIST_PRECISION': y[top].mean()}

best_cls = (cls_chosen.merge(cls_results[cls_results['TEST_YEAR'] == 2024], on=['ERA', 'MODEL', 'INPUTS', 'PARAMS'])
            .sort_values('AUC', ascending=False).groupby(['ERA', 'MODEL']).head(1))
cls_final, final_preds, cls_choice = [], {}, {}
for r in best_cls.itertuples():
    key = (r.ERA, r.MODEL, r.INPUTS, r.PARAMS)
    era_rows = df_features.loc[encoded[r.ERA].index]
    y_2024 = era_rows.loc[era_rows['PERIOD_START'].dt.year == 2024, cls_target].to_numpy()
    y_2025 = era_rows.loc[era_rows['PERIOD_START'].dt.year == 2025, cls_target].to_numpy()
    p_2024, p_2025 = cls_scores[key + (2024,)], cls_scores[key + (2025,)]
    grid = np.linspace(0.05, 0.95, 91)
    threshold = grid[np.argmax([f1_score(y_2024, (p_2024 >= t).astype(int)) for t in grid])]
    for label, t in [('Default 0.5', 0.5), ('Tuned on 2024', threshold)]:
        cls_final.append({'ERA': r.ERA, 'MODEL': r.MODEL, 'INPUTS': r.INPUTS, 'PARAMS': r.PARAMS, 'THRESHOLD_RULE': label,
                          **class_metrics(y_2025, p_2025, t)})
    final_preds[(r.ERA, r.MODEL)] = (y_2025, (p_2025 >= threshold).astype(int))
    cls_choice[(r.ERA, r.MODEL)] = (r.INPUTS, r.PARAMS, threshold)
cls_final = pd.DataFrame(cls_final)
cls_final.round(3)

In [ ]:
# confusion matrices on the 2025 test year at the tuned threshold (rows: actual, columns: predicted)
fig, axes = plt.subplots(2, len(era_order), figsize=(13, 7))
for j, era in enumerate(era_order):
    for i, model in enumerate(['Gradient boosted trees', 'Logistic regression']):
        y, pred = final_preds[(era, model)]
        cm = confusion_matrix(y, pred)
        sns.heatmap(cm, annot=True, fmt=',', cmap='Blues', cbar=False, ax=axes[i, j],
                    xticklabels=['Not', 'Outperformer'], yticklabels=['Not', 'Outperformer'])
        axes[i, j].set_title(f'{era}: {model}', fontsize=10)
        axes[i, j].set_xlabel('Predicted'); axes[i, j].set_ylabel('Actual')
fig.suptitle('Outperformer Classification on the 2025 Test Year (false positives = Type I, false negatives = Type II)')
plt.tight_layout()
plt.show()

**Observations -**

- **Refinement:** input sets moved tree AUC very little (2025 AUC 0.917 to 0.921 in 1981 to 1999), but specifications only lowered logistic regression's AUC (0.884 to 0.834 in 1981 to 1999, 0.978 to 0.927 in 2000 to 2009). On 2024, trees preferred the slower learning rate (0.03), and logistic regression preferred light regularization (C = 10 in most cases).
- **Threshold tuning on 2024 traded precision for recall.** For example, logistic regression in 1981 to 1999 went from recall 0.45 to 0.69 and F1 0.60 to 0.68, and trees in 2010 onward from recall 0.65 to 0.83 and F1 0.69 to 0.71.
- **Verdict: gradient boosted trees performed better.** Trees have the higher AUC in every era (0.919 vs 0.884, 0.983 vs 0.978, 0.935 vs 0.903) and higher F1 at the tuned threshold (0.70 vs 0.68, 0.85 vs 0.84, 0.71 vs 0.66). The gap is smallest in 2000 to 2009, where both models are very accurate.
- **Errors in business terms.** At the tuned threshold, trees make 126, 148, and 455 false positives (Type I: flagged but not outperformers) and 303, 45, and 149 false negatives (Type II: missed outperformers). Their 10% highest scores are 93%, 99%, and 83% true outperformers (logistic regression: 89%, 96%, and 80%), so a tree based watch list would be mostly correct.

### Generalization to Unseen Model Families

A different train and test split: 20% of model families (every model year of a make and model) are held out of training entirely, to test whether the chosen models carry over to vehicles they have never seen.

In [ ]:
# refinement by split: hold out 20% of model families (every model year of a make and model) entirely, train on the
# other families through 2024, and test on the held out families in 2025. A cohort's excess attrition is very
# persistent, so this checks that the models learn general patterns rather than recognizing cohorts they have seen
families = df_features['MAKE'] + ' ' + df_features['MODEL']
held_out = families.isin(np.random.default_rng(0).choice(families.unique(), size=int(families.nunique() * 0.2), replace=False))
d = df_features.sort_values(cohort_keys + ['PERIOD_START'])
stay = d.loc[d.groupby(cohort_keys)['OUTPERFORMER'].shift() == 1, 'OUTPERFORMER'].mean()
print(f"outperformers that are outperformers again the next year: {stay:.1%} (base rate 20%)")

gen_rows = []
for task, choice, models, target, top_key in [('Regression: MAE (points)', reg_choice, reg_models, reg_target, 'ANNUAL_ATTRITION'),
                                              ('Classification: AUC', cls_choice, cls_models, cls_target, 'EXCESS_ATTRITION')]:
    for (era, name), cfg in choice.items():
        features = input_sets(era, importance[(era, top_key)].nlargest(10).index.tolist())[cfg[0]]
        X, rows, _, _ = split(era, features, 2025)
        year, unseen = rows['PERIOD_START'].dt.year, held_out[X.index]
        for split_name, train, test in [('Seen cohorts (2025)', year < 2025, year == 2025),
                                        ('Unseen model families (2025)', (year < 2025) & ~unseen, (year == 2025) & unseen)]:
            model = models[name](**ast.literal_eval(cfg[1])).fit(X[train], rows.loc[train, target])
            y = rows.loc[test, target]
            score = (mean_absolute_error(y, model.predict(X[test])) * 100 if task.startswith('Regression')
                     else roc_auc_score(y, model.predict_proba(X[test])[:, 1]))
            gen_rows.append({'TASK': task, 'ERA': era, 'MODEL': name, 'SPLIT': split_name, 'SCORE': score, 'TEST_ROWS': int(test.sum())})
gen_results = pd.DataFrame(gen_rows)
gen_results.pivot_table(index=['TASK', 'ERA', 'MODEL'], columns='SPLIT', values='SCORE', observed=True).round(3)

**Observations -**

- Outperformers repeat strongly (70.6% are outperformers again the next year, against a 20% base rate), so models could score well partly by recognizing cohorts they have already seen.
- On model families never seen in training, performance drops but holds: tree AUC is 0.888, 0.979, and 0.935 (0.919, 0.983, and 0.935 on seen cohorts), and tree MAE is 1.87, 1.30, and 0.99 points (1.64, 1.09, and 0.84).
- Trees still beat the linear models on unseen families, except 1981 to 1999 classification, where logistic regression matches them (AUC 0.886 vs 0.888). The models therefore learn patterns that transfer to new vehicles, which matters for spotting emerging collectibles that have no history of their own yet.

### Vehicle Clustering (RQ1)

K-means against hierarchical clustering (Ward linkage) on era relative specification features, separately for cars and trucks, refined over 2 to 6 clusters and judged by silhouette (higher is better), Davies-Bouldin (lower is better), and Calinski-Harabasz (higher is better).

In [ ]:
# RQ1 clustering: group cohorts by specification profile only (no attrition), separately for cars and trucks,
# using era relative features so clusters reflect vehicle type rather than model year
clu = specs.copy()
clu['DISP_RANK'] = clu.groupby(['YEAR_MODEL', 'VEHICLE_CLASS'])['DISP_PER_1000_LBS'].rank(pct=True)
clu['LOG_PRICE_SPREAD'] = np.log(clu['PRICE_SPREAD'])
clu['IS_SPORT_SEGMENT'] = (clu['SEGMENT_FAMILY'] == 'Sport').astype(int)
clu['IS_LUXURY_SEGMENT'] = clu['SEGMENT_FAMILY'].isin(['Luxury', 'Luxury Utility']).astype(int)
clu['IS_COUPE_CONV'] = clu['BODY_FAMILY'].isin(['Coupe', 'Convertible']).astype(int)
cluster_features = ['MSRP_RANK', 'LOG_PRICE_SPREAD', 'DISP_RANK', 'MANUAL_VS_ERA', 'RWD_VS_ERA', 'HAS_AWD', 'HAS_4WD',
                    'HAS_CONVERTIBLE', 'HAS_REMOVABLE_ROOF', 'IS_SPORT_SEGMENT', 'IS_LUXURY_SEGMENT', 'IS_COUPE_CONV',
                    'LOG_CONFIG_COUNT']
# unknown flags count as typical for their era (0 after the era adjustment, or 0 for yes/no flags)
clu[cluster_features] = clu[cluster_features].fillna(0)

# refinement: 2 to 6 clusters, k-means against hierarchical clustering (Ward linkage), scored on the same inputs
clu_rows, clu_labels = [], {}
for cls in class_order:
    Z = StandardScaler().fit_transform(clu.loc[clu['VEHICLE_CLASS'] == cls, cluster_features])
    for k in range(2, 7):
        for name, model in [('K-means', KMeans(n_clusters=k, n_init=10, random_state=0)),
                            ('Hierarchical (Ward)', AgglomerativeClustering(n_clusters=k, linkage='ward'))]:
            labels = model.fit_predict(Z)
            clu_labels[(cls, name, k)] = labels
            clu_rows.append({'CLASS': cls, 'ALGORITHM': name, 'K': k,
                             'SILHOUETTE': silhouette_score(Z, labels, sample_size=4000, random_state=0),
                             'DAVIES_BOULDIN': davies_bouldin_score(Z, labels), 'CALINSKI_HARABASZ': calinski_harabasz_score(Z, labels),
                             'SMALLEST_CLUSTER_PCT': np.bincount(labels).min() / len(labels) * 100})
clu_results = pd.DataFrame(clu_rows)
clu_results.round(3)

In [ ]:
# chosen solution per class: the highest silhouette among solutions whose smallest cluster holds at least 5% of cohorts
eligible = clu_results[clu_results['SMALLEST_CLUSTER_PCT'] >= 5]
chosen_clu = eligible.loc[eligible.groupby('CLASS')['SILHOUETTE'].idxmax()].set_index('CLASS')
display(chosen_clu.round(3))

# clusters are named from their profiles: for cars the cluster with more sport segment models is enthusiast oriented;
# trucks split by drivetrain and body (RWD and 4x4 pickups and large SUVs versus AWD crossovers), so they are named that way
clu['CLUSTER'] = ''
for cls, r in chosen_clu.iterrows():
    m = clu['VEHICLE_CLASS'] == cls
    labels = pd.Series(clu_labels[(cls, r['ALGORITHM'], r['K'])], index=clu.index[m])
    means = clu.loc[m].groupby(labels)[cluster_features].mean()
    if cls == 'Car':
        lead = means['IS_SPORT_SEGMENT'].idxmax()
        names = {k: 'Car: enthusiast oriented' if k == lead else 'Car: transportation' for k in means.index}
    else:
        lead = means['HAS_4WD'].idxmax()
        names = {k: 'Truck: RWD and 4x4' if k == lead else 'Truck: AWD crossover' for k in means.index}
    clu.loc[m, 'CLUSTER'] = labels.map(names)

for name, g in clu.groupby('CLUSTER'):
    print(f"{name} ({len(g):,} cohorts): most common models", ', '.join((g['MAKE'] + ' ' + g['MODEL']).value_counts().head(6).index.str.title()))
profile = clu.groupby('CLUSTER')[cluster_features].mean().T
profile.round(2)

In [ ]:
# RQ1: representative attrition patterns by cluster, using the attrition periods from Feature Engineering
by_cluster = df_attrition.merge(clu[cohort_keys + ['CLUSTER']], on=cohort_keys)
cluster_colors = {'Car: enthusiast oriented': '#2a78d6', 'Car: transportation': '#9fc3ee',
                  'Truck: RWD and 4x4': '#eb6834', 'Truck: AWD crossover': '#f5b896'}
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, cls in zip(axes, class_order):
    for name in [n for n in cluster_colors if n.startswith(cls)]:
        curve = by_cluster[by_cluster['CLUSTER'] == name].groupby('AGE')['ANNUAL_ATTRITION'].median() * 100
        ax.plot(curve.index, curve.values, color=cluster_colors[name], linewidth=2, label=name.split(': ')[1])
    ax.set_title(cls)
    ax.set_xlabel('Vehicle Age at Period Start (years)')
    ax.legend(frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Median Annual Attrition (%)')
fig.suptitle('Representative Attrition Patterns by Specification Cluster (RQ1)')
plt.tight_layout()
plt.show()

# median excess attrition (points) by cluster and era; negative = losing vehicles more slowly than peers
(by_cluster.assign(ERA=pd.cut(by_cluster['YEAR_MODEL'], [1980, 1999, 2009, 2100], labels=era_order))
 .groupby(['CLUSTER', 'ERA'], observed=True)['EXCESS_ATTRITION'].median().unstack().mul(100).round(2))

**Observations -**

- Silhouette scores are modest (0.17 to 0.25), so specification profiles overlap rather than forming sharply separated groups. Once very small clusters (under 5% of cohorts) are ruled out, two clusters score best for both classes.
- **Verdict:** for cars, hierarchical clustering edges out k-means (silhouette 0.253 vs 0.249, Davies-Bouldin 1.84 vs 1.86, although k-means has the higher Calinski-Harabasz score); the two methods find nearly the same split. For trucks, k-means wins on all three measures (0.197 vs 0.178, 1.95 vs 2.13, and 1,130 vs 959).
- Cars split into **enthusiast oriented** (1,938 cohorts: 98% sport segment, 84% coupe or convertible, more RWD and manual availability than their era; e.g. Mustang, Corvette, 911, Camaro, Miata, GTI) and **transportation** (6,297 cohorts: mostly sedans in mainstream and luxury segments; e.g. Jetta, Accord, Corolla, Camry). Trucks split by drivetrain and body rather than enthusiasm: **RWD and 4x4** (3,016 cohorts: pickups, large SUVs, vans) and **AWD crossover** (2,756 cohorts: crossovers and luxury utilities).
- **RQ1 answer:** enthusiast oriented cars follow a much flatter attrition curve, peaking at 7.7% at age 26, against 17.1% at age 22 for transportation cars (5.9% vs 14.8% at ages 20 to 30), and the two converge after about age 36. Their excess attrition is negative in every era (-1.9, -5.6, and -0.9 points), while transportation cars run above the norm (+0.6, +1.5, and +0.2). Among trucks, RWD and 4x4 models peak at 9.2% against 15.2% for AWD crossovers.

## Model Selection Summary

| Task | Models compared | Selected | Why |
| --- | --- | --- | --- |
| Attrition regression (RQ2) | Gradient boosted trees, ridge regression | Gradient boosted trees | Lowest 2025 error in every era (1.64, 1.09, and 0.84 points against 1.90, 1.47, and 1.09 for ridge), stable across the 2024 and 2025 test years, and still ahead on unseen model families |
| Collectability classification | Gradient boosted trees, logistic regression | Gradient boosted trees | Higher AUC (0.919, 0.983, and 0.935) and F1 (0.70, 0.85, and 0.71) in every era, and a watch list that is 83% to 99% correct; logistic regression is a close, more interpretable alternative in 2000 to 2009 |
| Vehicle clustering (RQ1) | K-means, hierarchical (Ward) | Hierarchical for cars, k-means for trucks (two clusters each) | Best silhouette with no tiny clusters; cars separate into enthusiast oriented and transportation groups with clearly different attrition curves |

**How this addresses the business problem:** the regression model predicts next year's attrition for any cohort (RQ2), and the classifier ranks cohorts by their chance of outlasting their peers, which can become a watch list of emerging collectibles. The clustering confirms that enthusiast oriented cars lose vehicles at less than half the rate of transportation cars at their peak attrition ages.

**Limitations:**

- Only five annual periods exist, so models are tested on two future years (2024 and 2025).
- Relationships are associations, not causes.
- Cohorts below 100 vehicles are excluded, which underrepresents exotics.
- Cluster structure is modest (silhouette at most 0.25).
- The truck clusters reflect platform rather than enthusiasm.

## How the Business Would Use These Models

Three questions, three models. All of them score a **cohort** (one model year, make, and model), never an individual vehicle.

| Question | Model | Output |
| --- | --- | --- |
| How many of these will still be registered next year? | Attrition regression (gradient boosted trees) | Predicted annual attrition rate, accurate to about 1 point |
| Is this cohort outlasting comparable vehicles? | Outperformer classifier (gradient boosted trees) | Probability from 0 to 1, plus a ranked watch list |
| Is this an enthusiast or a transportation vehicle? | Specification clustering (hierarchical for cars) | One of two groups, from specifications alone |

### Scoring a cohort

1. **Check it qualifies.** The cohort must be a car or light truck, model year 1981 or later, with VINtelligence specifications, at least **3 years old**, and at least **100 registered vehicles** at the start of the period. Younger or smaller cohorts give unstable rates and are excluded.
2. **Pick the era model by model year:** 1981 to 1999, 2000 to 2009, or 2010 onward. Horsepower and related specifications only exist from model year 2000, so the earlier model uses fewer inputs.
3. **Supply only what is known at the start of the period:** age, registrations, decline from peak, last year's attrition and excess attrition, the registration shares (convertible, coupe, regular cab, work body, personal use), and the cohort's specifications.
4. **Read the results together.** Attrition answers *how fast*; the outperformer probability answers *whether that is unusual for its age*; the cluster answers *what kind of vehicle it is*.

### Reading the output

- **Predicted attrition** is a rate, not a count. A 6% prediction on 40,000 vehicles means roughly 2,400 fewer registrations next year.
- **Outperformer probability** ranks candidates. Use the ranking, not the raw number: the top 10% of scored cohorts are 83% to 99% true outperformers depending on era, so a short list is far more reliable than any single score.
- **Excess attrition, not raw attrition, is the signal.** Every cohort slows down with age, so only slowing *faster than peers of the same class, age, and year* is meaningful.
- **Enthusiast cars behave differently throughout life:** attrition peaks at about 7.7% at age 26, against 17.1% at age 22 for transportation cars. A cohort in the enthusiast cluster is expected to decline slowly; a *transportation* cohort that behaves like an enthusiast one is the more interesting find.

### Where it fits

- **Emerging collectability watch list.** Score every eligible cohort, filter to the enthusiast cluster or to transportation cohorts with negative excess attrition, and rank by outperformer probability. Model years 2000 to 2009 are the most productive hunting ground today: they are at peak attrition age, and the separation between cohorts is widest there.
- **Market sizing and forecasting.** Apply predicted attrition to current registrations to project the insurable population of a cohort, segment, or era.
- **Review and prioritization, not pricing.** The output is a ranked candidate list for a human to review, not a valuation.

### Keeping it current

- **Refresh yearly, after the April VIO snapshot loads.** The annual periods run April to April, so a new April snapshot adds one period for every cohort. Rerun the pipeline, retrain on all periods through the new year, and test on it before trusting the scores.
- **Retrain, do not extend.** New model years enter the 2010 onward era as they reach age 3, which shifts that era's mix; refitting keeps it calibrated.
- **Watch for reclassifications.** The snapshot coverage check catches registrations moving between models (as the 2007 Subaru Legacy and Outback did in April 2024). Run it after every load.

### Limits to respect

- Only **five annual periods** exist, so the models were tested on two future years and nothing longer.
- Relationships are **associations, not causes**: a convertible surviving well does not prove the body style caused it.
- **Exotics are underrepresented**, because cohorts below 100 vehicles are excluded. Vehicles that are collectible by design need separate handling.
- A cohort **blends its versions.** A Mustang cohort mixes base and performance cars, so scores describe the model as a whole.
- **Truck clusters split by platform** (rear drive and 4x4 against all wheel drive crossovers), not by enthusiasm, so the enthusiast grouping applies to cars only.